<a href="https://colab.research.google.com/github/poornikajalavadi/Generative-Project-Neural-Networks/blob/main/Generative_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Milestone 1

In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Install dependencies
!pip install -q torch torchvision
!pip install -q transformers
!pip install -q pycocotools pillow tqdm matplotlib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 50)
print("1. PyTorch")
print("=" * 50)
print(f"   Version:  {torch.__version__}")
print(f"   Device:   {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU:      {torch.cuda.get_device_name(0)}")
    print(f"   Memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")


# Hugging Face Transformers setup
import transformers
from transformers import GPT2Tokenizer, GPT2LMHeadModel

print("\n" + "=" * 50)
print("2. Hugging Face Transformers")
print("=" * 50)
print(f"   Version:  {transformers.__version__}")

#load GPT-2 tokenizer
_test_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
print(f"   GPT-2 tokenizer: OK (vocab size: {_test_tokenizer.vocab_size})")
del _test_tokenizer


#CLIP setup (via Hugging Face)
from transformers import CLIPModel, CLIPProcessor

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

print("\n" + "=" * 50)
print("3. CLIP (via Hugging Face)")
print("=" * 50)
print(f"   Model:    {CLIP_MODEL_NAME}")

#load CLIP processor
_test_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
print(f"   Processor: OK")
print(f"   Image size: {_test_processor.image_processor.size}")
del _test_processor


#Supporting libraries
import os
import json
import random
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from pycocotools.coco import COCO
import matplotlib.pyplot as plt

print("\n" + "=" * 50)
print("4. Supporting Libraries")
print("=" * 50)
print(f"   PIL (Pillow):   OK")
print(f"   pycocotools:    OK")
print(f"   matplotlib:     OK")


#Constants & reproducibility
SEED = 42
SUBSET_SIZE = 5000
MAX_CAPTION_LEN = 64

random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/image-captioning/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 50)
print("5. Configuration")
print("=" * 50)
print(f"   Seed:             {SEED}")
print(f"   Subset size:      {SUBSET_SIZE}")
print(f"   Max caption len:  {MAX_CAPTION_LEN} tokens")
print(f"   Data directory:   {DATA_DIR}")
print("\n Finished")

Download , clean and Preprocess the dataset

In [ ]:

#DOWNLOAD & PREPROCESS COCO CAPTIONS


import os
import random
from pathlib import Path
from pycocotools.coco import COCO
from google.colab import drive

#Download images & annotations to Google Drive

# Force remount to fix potential IOError
drive.mount("/content/drive", force_remount=True)

DATA_DIR = Path("/content/drive/MyDrive/image-captioning/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

IMG_DIR = DATA_DIR / "train2017"
ANN_DIR = DATA_DIR / "annotations"

if not IMG_DIR.exists():
    print("Downloading COCO train2017 images (this takes ~15-20 min)...")
    os.system("wget -q http://images.cocodataset.org/zips/train2017.zip -P /content/")
    os.system(f"unzip -q /content/train2017.zip -d {DATA_DIR}/")
    os.system("rm /content/train2017.zip")  # free up space
    print("Images downloaded & extracted.")
else:
    print(f"Images already exist at {IMG_DIR}, skipping download.")

if not ANN_DIR.exists():
    print("Downloading COCO annotations...")
    os.system("wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P /content/")
    os.system(f"unzip -q /content/annotations_trainval2017.zip -d {DATA_DIR}/")
    os.system("rm /content/annotations_trainval2017.zip")
    print("Annotations downloaded & extracted.")
else:
    print(f"Annotations already exist at {ANN_DIR}, skipping download.")

ANN_FILE = ANN_DIR / "captions_train2017.json"

# Make sure everything downloaded correctly
if not ANN_FILE.exists():
    raise FileNotFoundError(f"Annotations not found at {ANN_FILE}")
if not IMG_DIR.exists():
    raise FileNotFoundError(f"Images not found at {IMG_DIR}")


#Load annotations & sample 5k unique images
coco = COCO(str(ANN_FILE))
all_img_ids = list(coco.imgs.keys())
print(f"Total images in COCO train2017: {len(all_img_ids)}")

# Constants needed if previous cell wasn't run
SUBSET_SIZE = 5000
SEED = 42
random.seed(SEED)

# Randomly pick 5000 images
sampled_ids = sorted(random.sample(all_img_ids, SUBSET_SIZE))
print(f"Sampled subset: {len(sampled_ids)} images")


# Build (image_path, caption) pairs
# For each sampled image, grab the first caption and pair them together.
pairs = []
skipped = 0

print("Building dataset pairs...")
for img_id in sampled_ids:
    # Get image filename from COCO metadata
    info = coco.imgs[img_id]
    fname = info["file_name"]
    path = IMG_DIR / fname

    # Skip if the image file is missing
    # Wrapped in try-except to handle transient IOErrors from Drive
    try:
        if not path.exists():
            skipped += 1
            continue
    except OSError as e:
        # If Drive is flaky, we might just skip this one or retry
        print(f"Warning: IO Error accessing {fname}: {e}")
        skipped += 1
        continue

    # Look up captions for this image and take the first one
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)
    caption = anns[0]["caption"].strip().lower()

    pairs.append({
        "image_id": img_id,
        "path": str(path),
        "caption": caption
    })

print(f"\nValid image–caption pairs: {len(pairs)}")
print(f"Skipped (missing files):   {skipped}")
print(f"\n--- First 3 examples ---")
for i, p in enumerate(pairs[:3]):
    print(f"  [{i}] Image: {Path(p['path']).name}")
    print(f"       Caption: {p['caption']}\n")

In [ ]:
import matplotlib.pyplot as plt

def show_pairs(pairs, n=5):
    """Display n images with their captions side by side."""
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    for i, ax in enumerate(axes):
        img = Image.open(pairs[i]["path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(pairs[i]["caption"], fontsize=9, wrap=True)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_pairs(pairs, n=5)

CLIP IMAGE EMBEDDING PIPELINE


In [ ]:

#CLIP IMAGE EMBEDDING PIPELINE

#Load CLIP model & processor


import torch
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import CLIPModel, CLIPProcessor

# Define constants locally in case previous cells were skipped
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model.eval()  # inference mode — no dropout, no training

EMB_DIM = clip_model.config.projection_dim  # 512

print(f"CLIP model loaded: {CLIP_MODEL_NAME}")
print(f"Running on: {DEVICE}")
print(f"Embedding dimension: {EMB_DIM}")


#Encode all 5k images into embeddings
@torch.no_grad()  # skip gradient tracking — saves memory & speeds up
def encode_images(pairs, batch_size=64):

    all_embs = []

    for i in tqdm(range(0, len(pairs), batch_size), desc="CLIP encoding"):
        # Grab a batch of file paths
        batch_paths = [p["path"] for p in pairs[i : i + batch_size]]

        # Open each image as RGB
        images = [Image.open(fp).convert("RGB") for fp in batch_paths]

        # CLIPProcessor handles all preprocessing automatically:
        #   - Resize to 224x224
        #   - Center crop
        #   - Convert to tensor
        #   - Normalize with CLIP's mean & std values
        inputs = clip_processor(images=images, return_tensors="pt").to(DEVICE)

        # Pass through CLIP's vision encoder → (batch_size, 512)
        outputs = clip_model.get_image_features(**inputs)

        # Some versions return a tensor directly, others return an object
        # This handles both cases safely
        embs = outputs if isinstance(outputs, torch.Tensor) else outputs.pooler_output

        # L2 normalize: scale each vector to length 1
        # After this, cosine similarity = simple dot product
        embs = embs / embs.norm(dim=-1, keepdim=True)

        # Move to CPU to free up GPU memory for next batch
        all_embs.append(embs.cpu())

    # Stack all batches into one tensor
    return torch.cat(all_embs, dim=0)

print(f"\nEncoding {len(pairs)} images...")
image_embeddings = encode_images(pairs)
print(f"Done! Embeddings shape: {image_embeddings.shape}")  # (5000, 512)


# Sanity checks
# Test 1: An image compared to itself should have similarity = 1.0
# Test 2: Two different images should have similarity < 1.0

cos = torch.nn.CosineSimilarity(dim=0)

sim_same = cos(image_embeddings[0], image_embeddings[0]).item()
sim_diff = cos(image_embeddings[0], image_embeddings[1]).item()

print(f"\nSanity checks:")
print(f"  Image 0 vs itself:   {sim_same:.4f}  ✅ (expect 1.0)")
print(f"  Image 0 vs Image 1:  {sim_diff:.4f}  ✅ (expect < 1.0)")

# Test 3: Check embedding statistics
print(f"\nEmbedding stats:")
print(f"  Mean:  {image_embeddings.mean().item():.6f}")
print(f"  Std:   {image_embeddings.std().item():.6f}")
print(f"  Min:   {image_embeddings.min().item():.6f}")
print(f"  Max:   {image_embeddings.max().item():.6f}")


#Visualize: similarity search
# Pick one image, find the 3 most similar images in our dataset
# based on CLIP embeddings. This proves the embeddings capture
# meaningful visual information.

def show_similar_pair(embeddings, pairs, idx=0, top_k=3):
    """Find and display the most similar images to pairs[idx]."""
    # Compare image[idx] against all other images
    sims = torch.nn.functional.cosine_similarity(
        embeddings[idx].unsqueeze(0), embeddings, dim=1
    )
    # Exclude self-match
    sims[idx] = -1
    top_indices = sims.argsort(descending=True)[:top_k]

    # Plot query + matches
    fig, axes = plt.subplots(1, top_k + 1, figsize=(4 * (top_k + 1), 4))

    # Query image (left)
    query_img = Image.open(pairs[idx]["path"]).convert("RGB")
    axes[0].imshow(query_img)
    axes[0].set_title(f"QUERY\n{pairs[idx]['caption'][:50]}...", fontsize=9)
    axes[0].axis("off")

    # Matched images (right)
    for j, match_idx in enumerate(top_indices):
        match_img = Image.open(pairs[match_idx]["path"]).convert("RGB")
        axes[j + 1].imshow(match_img)
        sim_score = sims[match_idx].item()
        axes[j + 1].set_title(
            f"Match {j+1} (sim={sim_score:.3f})\n{pairs[match_idx]['caption'][:50]}...",
            fontsize=9
        )
        axes[j + 1].axis("off")

    plt.suptitle("CLIP Embedding Similarity Search", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

# Try a few different query images
print("\n--- Similarity search examples ---")
show_similar_pair(image_embeddings, pairs, idx=0)
show_similar_pair(image_embeddings, pairs, idx=100)

In [ ]:

#GPT-2 CAPTION TOKENIZATION



from transformers import GPT2Tokenizer

#Load tokenizer & add special tokens
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 doesn't have a pad token by default, so we add three
# special tokens for our captioning task
special_tokens = {
    "bos_token": "<|startofcaption|>",   # beginning of sequence
    "eos_token": "<|endofcaption|>",     # end of sequence
    "pad_token": "<|pad|>",              # padding
}
num_added = tokenizer.add_special_tokens(special_tokens)

print(f"GPT-2 base vocab size:  50257")
print(f"Special tokens added:   {num_added}")
print(f"New vocab size:         {len(tokenizer)}")
print(f"  START token: '{tokenizer.bos_token}' → ID {tokenizer.bos_token_id}")
print(f"  END token:   '{tokenizer.eos_token}' → ID {tokenizer.eos_token_id}")
print(f"  PAD token:   '{tokenizer.pad_token}' → ID {tokenizer.pad_token_id}")


#Tokenize all captions


MAX_CAPTION_LEN = 64  # max tokens per caption (including special tokens)

import torch

all_input_ids = []
all_attention_masks = []

print(f"\nTokenizing {len(pairs)} captions (max length: {MAX_CAPTION_LEN})...")

for i, pair in enumerate(tqdm(pairs, desc="Tokenizing")):
    # Wrap caption with special tokens
    text = f"{tokenizer.bos_token} {pair['caption']} {tokenizer.eos_token}"

    # Tokenize — this converts text → list of integer IDs
    encoded = tokenizer(
        text,
        max_length=MAX_CAPTION_LEN,
        padding="max_length",          # pad short captions to MAX_CAPTION_LEN
        truncation=True,               # cut long captions at MAX_CAPTION_LEN
        return_tensors="pt",           # return PyTorch tensors
        return_attention_mask=True,
    )

    all_input_ids.append(encoded["input_ids"].squeeze(0))          # shape: (64,)
    all_attention_masks.append(encoded["attention_mask"].squeeze(0))  # shape: (64,)

# Stack into single tensors
caption_input_ids = torch.stack(all_input_ids)          # (5000, 64)
caption_attention_masks = torch.stack(all_attention_masks)  # (5000, 64)

print(f"\nTokenization complete!")
print(f"  Input IDs shape:      {caption_input_ids.shape}")
print(f"  Attention masks shape: {caption_attention_masks.shape}")


#Sanity checks
print("\n--- Tokenization Examples ---")
for i in range(3):
    ids = caption_input_ids[i]
    mask = caption_attention_masks[i]
    real_tokens = mask.sum().item()  # count of non-padding tokens

    # Decode back to text to verify it looks right
    decoded = tokenizer.decode(ids[: real_tokens], skip_special_tokens=False)

    print(f"\n  [{i}] Original:  {pairs[i]['caption']}")
    print(f"      Tokens:    {real_tokens} real + {MAX_CAPTION_LEN - real_tokens} padding = {MAX_CAPTION_LEN} total")
    print(f"      IDs (first 10): {ids[:10].tolist()}")
    print(f"      Mask(first 10): {mask[:10].tolist()}")
    print(f"      Decoded:   {decoded}")

# Distribution of caption lengths (in tokens)
real_lengths = caption_attention_masks.sum(dim=1)  # how many real tokens per caption
print(f"\n--- Caption Length Stats (in tokens) ---")
print(f"  Min:    {real_lengths.min().item()}")
print(f"  Max:    {real_lengths.max().item()}")
print(f"  Mean:   {real_lengths.float().mean().item():.1f}")
print(f"  Median: {real_lengths.float().median().item():.1f}")

# Check how many captions got truncated
truncated = (real_lengths == MAX_CAPTION_LEN).sum().item()
print(f"  Truncated (hit max): {truncated} ({truncated/len(pairs)*100:.1f}%)")

print("\n" + "=" * 55)
print("Captions are now token IDs.")
print("=" * 55)




In [ ]:

#BUNDLE & SAVE EVERYTHING TO DISK


import json
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/image-captioning/processed")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

#Save tensors (PyTorch format)
tensor_path = SAVE_DIR / "milestone1_tensors.pt"

save_data = {
    "image_embeddings": image_embeddings,            # (5000, 512)
    "caption_input_ids": caption_input_ids,          # (5000, 64)
    "caption_attention_masks": caption_attention_masks,  # (5000, 64)
    "vocab_size": len(tokenizer),                    # 50260
    "max_caption_len": MAX_CAPTION_LEN,              # 64
    "clip_emb_dim": image_embeddings.shape[1],       # 512
}

torch.save(save_data, tensor_path)
print(f"Tensors saved to: {tensor_path}")
print(f"  File size: {tensor_path.stat().st_size / 1e6:.1f} MB")

#Save metadata (JSON format)
meta_path = SAVE_DIR / "milestone1_metadata.json"

metadata = {
    "num_pairs": len(pairs),
    "subset_size": SUBSET_SIZE,
    "seed": SEED,
    "max_caption_len": MAX_CAPTION_LEN,
    "clip_model": "openai/clip-vit-base-patch32",
    "clip_emb_dim": 512,
    "vocab_size": len(tokenizer),
    "special_tokens": special_tokens,
    "pairs": pairs,  # includes image_id, path, caption for each
}

with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {meta_path}")
print(f"  File size: {meta_path.stat().st_size / 1e6:.1f} MB")

#Save tokenizer (so we can reload it exactly)
tok_path = SAVE_DIR / "tokenizer"
tokenizer.save_pretrained(str(tok_path))
print(f"Tokenizer saved to: {tok_path}")

#Final verification
print("\n--- Verification: reload and check shapes ---")
loaded = torch.load(tensor_path, weights_only=True)
print(f"  image_embeddings:       {loaded['image_embeddings'].shape}")
print(f"  caption_input_ids:      {loaded['caption_input_ids'].shape}")
print(f"  caption_attention_masks: {loaded['caption_attention_masks'].shape}")
print(f"  vocab_size:             {loaded['vocab_size']}")

# Quick sanity: first caption should decode correctly
reloaded_tokenizer = GPT2Tokenizer.from_pretrained(str(tok_path))
first_ids = loaded["caption_input_ids"][0]
first_mask = loaded["caption_attention_masks"][0]
real_len = first_mask.sum().item()
decoded = reloaded_tokenizer.decode(first_ids[:real_len], skip_special_tokens=False)
print(f"\n  First caption decoded: {decoded}")
print(f"  Original caption:      {pairs[0]['caption']}")

print("\n" + "=" * 55)
print("MILESTONE 1 COMPLETE!")
print("=" * 55)
print(f"""
  Everything is saved to Google Drive:
    📁 {SAVE_DIR}/
       ├── milestone1_tensors.pt    (embeddings + tokens)
       ├── milestone1_metadata.json (paths, captions, config)
       └── tokenizer/               (GPT-2 + special tokens)

  For Milestone 2, just load with:
    data = torch.load('{tensor_path}')
    tokenizer = GPT2Tokenizer.from_pretrained('{tok_path}')
""")

Milestone 2

In [ ]:
# Cell 1 — Remount Google Drive
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Quick check that files are accessible
from pathlib import Path
IMG_DIR = Path("/content/drive/MyDrive/image-captioning/data/train2017")
test_files = list(IMG_DIR.iterdir())
print(f"✅ Found {len(test_files)} files in train2017/")

In [ ]:
%pip install open_clip_torch

In [ ]:
"""
CLIP Image Encoder + GPT-2 Language Decoder
Two injection strategies: Prefix Conditioning & Cross-Attention

Requirements:
    pip install torch transformers open-clip-torch pillow
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config
from transformers.modeling_outputs import CausalLMOutputWithCrossAttentions
import open_clip
from PIL import Image
from typing import Optional, Literal


# =============================================================================
# 1. CLIP Visual Encoder Wrapper
# =============================================================================

class CLIPVisualEncoder(nn.Module):
    """Wraps an OpenCLIP vision encoder and exposes both CLS and patch tokens."""

    def __init__(
        self,
        model_name: str = "ViT-B-32",
        pretrained: str = "openai",
        freeze: bool = True,
    ):
        super().__init__()
        clip_model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name, pretrained=pretrained
        )
        self.visual = clip_model.visual          # vision transformer
        self.embed_dim = clip_model.visual.output_dim  # e.g. 512

        if freeze:
            for p in self.visual.parameters():
                p.requires_grad = False

    @torch.no_grad()
    def forward(self, pixel_values: torch.Tensor):
        """
        Args:
            pixel_values: [B, 3, H, W] preprocessed images
        Returns:
            cls_token:    [B, D_clip]          – pooled CLS embedding
            patch_tokens: [B, N_patches, D_clip] – per-patch embeddings
        """
        # OpenCLIP ViT returns the CLS-pooled vector by default.
        # To get patch tokens we hook into the internals.
        features = self.visual(pixel_values)  # [B, D_clip]
        cls_token = features

        # Retrieve intermediate patch tokens (before final projection)
        # by running the transformer trunk manually:
        x = self.visual.conv1(pixel_values)                    # [B, D, GH, GW]
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)  # [B, N, D]
        x = torch.cat([
            self.visual.class_embedding.unsqueeze(0).expand(x.shape[0], -1, -1),
            x
        ], dim=1)                                              # [B, 1+N, D]
        x = x + self.visual.positional_embedding.unsqueeze(0)
        x = self.visual.ln_pre(x)
        x = self.visual.transformer(x)                         # [B, 1+N, D_inner]
        patch_tokens = self.visual.ln_post(x[:, 1:, :])       # [B, N, D_inner]

        return cls_token, patch_tokens


# =============================================================================
# 2. Projection Layers
# =============================================================================

class LinearProjection(nn.Module):
    """Projects CLIP embeddings into GPT-2's embedding space."""

    def __init__(self, clip_dim: int, gpt_dim: int, num_prefix_tokens: int = 8):
        super().__init__()
        self.num_prefix_tokens = num_prefix_tokens
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, gpt_dim * num_prefix_tokens),
            nn.GELU(),
            nn.LayerNorm(gpt_dim * num_prefix_tokens),
        )
        self.gpt_dim = gpt_dim

    def forward(self, cls_token: torch.Tensor) -> torch.Tensor:
        """
        Args:
            cls_token: [B, D_clip]
        Returns:
            prefix_embeds: [B, num_prefix_tokens, D_gpt]
        """
        out = self.proj(cls_token)                       # [B, num_prefix * D_gpt]
        return out.view(-1, self.num_prefix_tokens, self.gpt_dim)


class PatchProjection(nn.Module):
    """Projects per-patch CLIP features for cross-attention."""

    def __init__(self, clip_dim: int, gpt_dim: int):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, gpt_dim),
            nn.GELU(),
            nn.LayerNorm(gpt_dim),
        )

    def forward(self, patch_tokens: torch.Tensor) -> torch.Tensor:
        """[B, N, D_clip] -> [B, N, D_gpt]"""
        return self.proj(patch_tokens)


# =============================================================================
# 3. Cross-Attention GPT-2 Block (inserted between existing layers)
# =============================================================================

class VisualCrossAttentionLayer(nn.Module):
    """A single cross-attention layer that attends over visual tokens."""

    def __init__(self, gpt_dim: int, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.ln = nn.LayerNorm(gpt_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=gpt_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True,
        )
        self.gate = nn.Parameter(torch.zeros(1))  # learnable gating scalar

    def forward(
        self,
        hidden_states: torch.Tensor,
        visual_ctx: torch.Tensor,
        visual_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            hidden_states: [B, T, D]  – current GPT-2 hidden states
            visual_ctx:    [B, N, D]  – projected patch embeddings
        Returns:
            hidden_states: [B, T, D]  – with cross-attended visual info mixed in
        """
        residual = hidden_states
        normed = self.ln(hidden_states)
        attn_out, _ = self.cross_attn(
            query=normed, key=visual_ctx, value=visual_ctx,
            key_padding_mask=visual_mask,
        )
        return residual + self.gate * attn_out


# =============================================================================
# 4a. Prefix-Conditioned CLIP-GPT2
# =============================================================================

class CLIPGpt2Prefix(nn.Module):
    """
    Strategy 1 – Prefix Conditioning:
    Project CLIP CLS into K learned "visual prefix" tokens and prepend them
    to the GPT-2 input embedding sequence.
    """

    def __init__(
        self,
        clip_model: str = "ViT-B-32",
        clip_pretrained: str = "openai",
        gpt2_model: str = "gpt2",
        num_prefix_tokens: int = 8,
        freeze_clip: bool = True,
        freeze_gpt2: bool = False,
    ):
        super().__init__()
        # --- Sub-modules ---
        self.clip = CLIPVisualEncoder(clip_model, clip_pretrained, freeze=freeze_clip)
        self.gpt2 = GPT2LMHeadModel.from_pretrained(gpt2_model)
        self.tokenizer = GPT2Tokenizer.from_pretrained(gpt2_model)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        gpt_dim = self.gpt2.config.n_embd   # 768 for gpt2-base
        clip_dim = self.clip.embed_dim       # 512 for ViT-B-32

        self.prefix_proj = LinearProjection(clip_dim, gpt_dim, num_prefix_tokens)
        self.num_prefix = num_prefix_tokens

        if freeze_gpt2:
            for p in self.gpt2.parameters():
                p.requires_grad = False

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
    ):
        B, T = input_ids.shape

        # 1. Encode image -> prefix embeddings
        cls_token, _ = self.clip(pixel_values)
        prefix_embeds = self.prefix_proj(cls_token)        # [B, K, D]

        # 2. Get text token embeddings from GPT-2
        text_embeds = self.gpt2.transformer.wte(input_ids) # [B, T, D]

        # 3. Concatenate: [visual_prefix | text_tokens]
        inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)  # [B, K+T, D]

        # 4. Extend attention mask to cover prefix tokens
        prefix_mask = torch.ones(B, self.num_prefix, device=input_ids.device)
        if attention_mask is None:
            attention_mask = torch.ones(B, T, device=input_ids.device)
        attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        # 5. Extend labels: -100 for prefix positions (don't compute loss there)
        if labels is not None:
            prefix_labels = torch.full(
                (B, self.num_prefix), -100, dtype=torch.long, device=labels.device
            )
            labels = torch.cat([prefix_labels, labels], dim=1)

        # 6. Forward through GPT-2
        outputs = self.gpt2(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
        )
        return outputs

    @torch.no_grad()
    def generate(
        self,
        pixel_values: torch.Tensor,
        max_new_tokens: int = 50,
        temperature: float = 0.8,
        top_p: float = 0.9,
    ) -> list[str]:
        self.eval()
        B = pixel_values.shape[0]
        device = pixel_values.device

        cls_token, _ = self.clip(pixel_values)
        prefix_embeds = self.prefix_proj(cls_token)

        # Start with BOS token
        cur_ids = torch.full((B, 1), self.tokenizer.bos_token_id, device=device)
        generated = cur_ids

        for _ in range(max_new_tokens):
            text_embeds = self.gpt2.transformer.wte(generated)
            embeds = torch.cat([prefix_embeds, text_embeds], dim=1)

            outputs = self.gpt2(inputs_embeds=embeds)
            next_logits = outputs.logits[:, -1, :] / temperature

            # Top-p (nucleus) sampling
            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
            cumulative = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cumulative - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = -float("inf")
            probs = F.softmax(sorted_logits, dim=-1)
            sampled = torch.multinomial(probs, 1)
            next_token = sorted_idx.gather(-1, sampled)

            generated = torch.cat([generated, next_token], dim=1)
            if (next_token == self.tokenizer.eos_token_id).all():
                break

        return self.tokenizer.batch_decode(generated, skip_special_tokens=True)


# =============================================================================
# 4b. Cross-Attention CLIP-GPT2
# =============================================================================

class CLIPGpt2CrossAttention(nn.Module):
    """
    Strategy 2 – Cross-Attention Injection:
    Insert cross-attention layers into GPT-2 that attend over projected
    CLIP patch tokens at every N-th transformer block.
    """

    def __init__(
        self,
        clip_model: str = "ViT-B-32",
        clip_pretrained: str = "openai",
        gpt2_model: str = "gpt2",
        xattn_every: int = 2,
        freeze_clip: bool = True,
        freeze_gpt2_self_attn: bool = True,
    ):
        super().__init__()
        self.clip = CLIPVisualEncoder(clip_model, clip_pretrained, freeze=freeze_clip)
        self.gpt2 = GPT2LMHeadModel.from_pretrained(gpt2_model)
        self.tokenizer = GPT2Tokenizer.from_pretrained(gpt2_model)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        gpt_dim = self.gpt2.config.n_embd
        clip_inner = self.clip.visual.ln_post.normalized_shape[0]

        # Projection for patch tokens
        self.patch_proj = PatchProjection(clip_inner, gpt_dim)

        # Insert cross-attention layers after every N-th GPT-2 block
        n_layers = self.gpt2.config.n_layer
        self.xattn_layers = nn.ModuleDict()
        for i in range(n_layers):
            if (i + 1) % xattn_every == 0:
                self.xattn_layers[str(i)] = VisualCrossAttentionLayer(gpt_dim)

        # Optionally freeze GPT-2's self-attention (train only cross-attn + LM head)
        if freeze_gpt2_self_attn:
            for name, p in self.gpt2.named_parameters():
                if "lm_head" not in name:
                    p.requires_grad = False

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
    ):
        # 1. Encode image patches
        _, patch_tokens = self.clip(pixel_values)
        visual_ctx = self.patch_proj(patch_tokens)        # [B, N_patches, D_gpt]

        # 2. Embed text tokens
        hidden = self.gpt2.transformer.wte(input_ids)
        hidden = hidden + self.gpt2.transformer.wpe(
            torch.arange(input_ids.size(1), device=input_ids.device)
        )
        hidden = self.gpt2.transformer.drop(hidden)

        # 3. Run through GPT-2 blocks with interleaved cross-attention
        for i, block in enumerate(self.gpt2.transformer.h):
            outputs = block(hidden, attention_mask=attention_mask)
            hidden = outputs[0]

            # Cross-attend to visual context at designated layers
            if str(i) in self.xattn_layers:
                hidden = self.xattn_layers[str(i)](hidden, visual_ctx)

        hidden = self.gpt2.transformer.ln_f(hidden)

        # 4. LM head
        logits = self.gpt2.lm_head(hidden)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return CausalLMOutputWithCrossAttentions(loss=loss, logits=logits)

    @torch.no_grad()
    def generate(
        self,
        pixel_values: torch.Tensor,
        max_new_tokens: int = 50,
        temperature: float = 0.8,
        top_p: float = 0.9,
    ) -> list[str]:
        self.eval()
        B = pixel_values.shape[0]
        device = pixel_values.device

        _, patch_tokens = self.clip(pixel_values)
        visual_ctx = self.patch_proj(patch_tokens)

        cur_ids = torch.full((B, 1), self.tokenizer.bos_token_id, device=device)

        for _ in range(max_new_tokens):
            hidden = self.gpt2.transformer.wte(cur_ids)
            hidden = hidden + self.gpt2.transformer.wpe(
                torch.arange(cur_ids.size(1), device=device)
            )
            hidden = self.gpt2.transformer.drop(hidden)

            for i, block in enumerate(self.gpt2.transformer.h):
                outputs = block(hidden)
                hidden = outputs[0]
                if str(i) in self.xattn_layers:
                    hidden = self.xattn_layers[str(i)](hidden, visual_ctx)

            hidden = self.gpt2.transformer.ln_f(hidden)
            logits = self.gpt2.lm_head(hidden)[:, -1, :] / temperature

            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cumulative = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cumulative - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = -float("inf")
            probs = F.softmax(sorted_logits, dim=-1)
            next_token = sorted_idx.gather(-1, torch.multinomial(probs, 1))

            cur_ids = torch.cat([cur_ids, next_token], dim=1)
            if (next_token == self.tokenizer.eos_token_id).all():
                break

        return self.tokenizer.batch_decode(cur_ids, skip_special_tokens=True)


# =============================================================================
# 5. Training Utilities
# =============================================================================

def count_trainable_params(model: nn.Module) -> str:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return (
        f"Total: {total / 1e6:.1f}M | "
        f"Trainable: {trainable / 1e6:.1f}M ({100 * trainable / total:.1f}%)"
    )


def build_training_step(model, optimizer, scaler=None):
    """Returns a single training step closure with optional AMP."""

    def step(batch):
        pixel_values = batch["pixel_values"]
        input_ids = batch["input_ids"]
        attention_mask = batch.get("attention_mask")
        labels = batch["labels"]

        optimizer.zero_grad()

        if scaler is not None:
            with torch.amp.autocast("cuda"):
                outputs = model(pixel_values, input_ids, attention_mask, labels)
            scaler.scale(outputs.loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(pixel_values, input_ids, attention_mask, labels)
            outputs.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        return outputs.loss.item()

    return step


# =============================================================================
# 6. Demo / Quick Test
# =============================================================================

if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    strategy = "prefix"  # or "cross_attention"

    print(f"Building {strategy} model on {device}...")

    if strategy == "prefix":
        model = CLIPGpt2Prefix(
            num_prefix_tokens=8,
            freeze_clip=True,
            freeze_gpt2=False,
        ).to(device)
    else:
        model = CLIPGpt2CrossAttention(
            xattn_every=2,
            freeze_clip=True,
            freeze_gpt2_self_attn=True,
        ).to(device)

    print(count_trainable_params(model))

    # --- Fake inference pass with a random image ---
    dummy_img = torch.randn(1, 3, 224, 224, device=device)
    captions = model.generate(dummy_img, max_new_tokens=30)
    print(f"Generated (random image): {captions[0]}")

    # --- Real image example (uncomment) ---
    # img = Image.open("example.jpg")
    # pixel_values = model.clip.preprocess(img).unsqueeze(0).to(device)
    # captions = model.generate(pixel_values, max_new_tokens=50)
    # print(f"Caption: {captions[0]}")

    # --- Training skeleton ---
    # optimizer = torch.optim.AdamW(
    #     filter(lambda p: p.requires_grad, model.parameters()),
    #     lr=5e-5, weight_decay=0.01,
    # )
    # scaler = torch.amp.GradScaler("cuda")
    # train_step = build_training_step(model, optimizer, scaler)
    #
    # for epoch in range(num_epochs):
    #     for batch in dataloader:
    #         loss = train_step(batch)
    #         print(f"loss: {loss:.4f}")

In [ ]:
"""
Baseline Caption Generation — Run CLIP-GPT2 (untrained bridge) on sample images
from your COCO dataset (already downloaded in Milestone 1).

This establishes a BEFORE-TRAINING baseline so you can measure improvement after fine-tuning.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers.modeling_outputs import CausalLMOutputWithCrossAttentions
from PIL import Image
from pathlib import Path
from pycocotools.coco import COCO
import pandas as pd
import json
import time
import random
from datetime import datetime
from typing import Optional


# ─────────────────────────────────────────────────────────────────────────────
# 1. MODEL CLASSES (same as your main code)
# ─────────────────────────────────────────────────────────────────────────────

class CLIPVisualEncoder(nn.Module):
    def __init__(self, model_name="ViT-B-32", pretrained="openai", freeze=True):
        super().__init__()
        clip_model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name, pretrained=pretrained
        )
        self.visual = clip_model.visual
        self.embed_dim = clip_model.visual.output_dim

        if freeze:
            for p in self.visual.parameters():
                p.requires_grad = False

    @torch.no_grad()
    def forward(self, pixel_values):
        x = self.visual.conv1(pixel_values)
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        x = torch.cat([
            self.visual.class_embedding.unsqueeze(0).expand(x.shape[0], -1, -1), x
        ], dim=1)
        x = x + self.visual.positional_embedding.unsqueeze(0)
        x = self.visual.ln_pre(x)
        x = self.visual.transformer(x)
        patch_tokens = self.visual.ln_post(x[:, 1:, :])
        cls_token = x[:, 0, :] @ self.visual.proj
        return cls_token, patch_tokens


class LinearProjection(nn.Module):
    def __init__(self, clip_dim, gpt_dim, num_prefix_tokens=8):
        super().__init__()
        self.num_prefix_tokens = num_prefix_tokens
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, gpt_dim * num_prefix_tokens),
            nn.GELU(),
            nn.LayerNorm(gpt_dim * num_prefix_tokens),
        )
        self.gpt_dim = gpt_dim

    def forward(self, cls_token):
        out = self.proj(cls_token)
        return out.view(-1, self.num_prefix_tokens, self.gpt_dim)


class CLIPGpt2Prefix(nn.Module):
    def __init__(self, num_prefix_tokens=8, freeze_clip=True, freeze_gpt2=False):
        super().__init__()
        self.clip = CLIPVisualEncoder(freeze=freeze_clip)
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        self.tokenizer.pad_token = self.tokenizer.eos_token

        gpt_dim = self.gpt2.config.n_embd
        clip_dim = self.clip.embed_dim
        self.prefix_proj = LinearProjection(clip_dim, gpt_dim, num_prefix_tokens)
        self.num_prefix = num_prefix_tokens

        if freeze_gpt2:
            for p in self.gpt2.parameters():
                p.requires_grad = False

    @torch.no_grad()
    def generate(self, pixel_values, max_new_tokens=50, temperature=0.8, top_p=0.9):
        self.eval()
        B = pixel_values.shape[0]
        device = pixel_values.device

        cls_token, _ = self.clip(pixel_values)
        prefix_embeds = self.prefix_proj(cls_token)

        cur_ids = torch.full((B, 1), self.tokenizer.bos_token_id, device=device)
        generated = cur_ids

        for _ in range(max_new_tokens):
            text_embeds = self.gpt2.transformer.wte(generated)
            embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
            outputs = self.gpt2(inputs_embeds=embeds)
            next_logits = outputs.logits[:, -1, :] / temperature

            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
            cumulative = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cumulative - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = -float("inf")
            probs = F.softmax(sorted_logits, dim=-1)
            sampled = torch.multinomial(probs, 1)
            next_token = sorted_idx.gather(-1, sampled)

            generated = torch.cat([generated, next_token], dim=1)
            if (next_token == self.tokenizer.eos_token_id).all():
                break

        return self.tokenizer.batch_decode(generated, skip_special_tokens=True)


# ─────────────────────────────────────────────────────────────────────────────
# 2. PATHS — pointing to your Milestone 1 COCO data on Google Drive
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR = Path("/content/drive/MyDrive/image-captioning/data")
IMG_DIR  = DATA_DIR / "train2017"
ANN_FILE = DATA_DIR / "annotations" / "captions_train2017.json"

# Verify data exists
assert IMG_DIR.exists(),  f"❌ Image folder not found: {IMG_DIR}"
assert ANN_FILE.exists(), f"❌ Annotations not found: {ANN_FILE}"
print(f"✅ Using COCO data from: {DATA_DIR}")


# ─────────────────────────────────────────────────────────────────────────────
# 3. SAMPLE IMAGES FROM YOUR EXISTING DATASET
# ─────────────────────────────────────────────────────────────────────────────

SEED = 42
NUM_SAMPLES = 10  # number of images for baseline evaluation

random.seed(SEED)

# Load COCO annotations
coco = COCO(str(ANN_FILE))
all_img_ids = list(coco.imgs.keys())
print(f"Total images in COCO annotations: {len(all_img_ids)}")

# Filter to only images that actually exist on disk (your 5k subset)
available_ids = [
    img_id for img_id in all_img_ids
    if (IMG_DIR / coco.imgs[img_id]["file_name"]).exists()
]
print(f"Images available on disk: {len(available_ids)}")

# Sample from available images only
sample_ids = random.sample(available_ids, min(NUM_SAMPLES, len(available_ids)))

# Build (path, ground_truth_captions) for each sample
sample_data = []
for img_id in sample_ids:
    info = coco.imgs[img_id]
    fname = info["file_name"]
    path = IMG_DIR / fname

    if not path.exists():
        print(f"  ⚠️ Skipping {fname} — file not found")
        continue

    # Get ALL ground truth captions for this image
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)
    captions = [a["caption"].strip() for a in anns]

    sample_data.append({
        "image_id": img_id,
        "file_name": fname,
        "path": str(path),
        "ground_truth_captions": captions,
    })

print(f"Loaded {len(sample_data)} sample images for baseline evaluation\n")


# ─────────────────────────────────────────────────────────────────────────────
# 4. BUILD MODEL & RUN BASELINE CAPTIONING
# ─────────────────────────────────────────────────────────────────────────────

def run_baseline():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print("=" * 70)

    # Build model
    print("\n📦 Building CLIPGpt2Prefix model (untrained bridge)...")
    model = CLIPGpt2Prefix(
        num_prefix_tokens=8,
        freeze_clip=True,
        freeze_gpt2=False,
    ).to(device)

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Total: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M")
    print("=" * 70)

    # Generate captions for each sample image
    print("\n🖼️  Generating baseline captions...\n")
    results = []

    for i, item in enumerate(sample_data):
        print(f"── [{i+1}/{len(sample_data)}] {item['file_name']} ──")
        print(f"   Ground truth: {item['ground_truth_captions'][0]}")

        # Load and preprocess image using CLIP's preprocessor
        img = Image.open(item["path"]).convert("RGB")
        pixel_values = model.clip.preprocess(img).unsqueeze(0).to(device)

        start = time.time()

        # Generate with 3 decoding strategies
        caption_greedy = model.generate(
            pixel_values, max_new_tokens=40, temperature=0.1, top_p=1.0
        )[0]

        caption_sampled = model.generate(
            pixel_values, max_new_tokens=40, temperature=0.8, top_p=0.9
        )[0]

        caption_creative = model.generate(
            pixel_values, max_new_tokens=40, temperature=1.2, top_p=0.95
        )[0]

        elapsed = time.time() - start

        print(f"   Greedy:   {caption_greedy[:100]}")
        print(f"   Sampled:  {caption_sampled[:100]}")
        print(f"   Creative: {caption_creative[:100]}")
        print(f"   Time: {elapsed:.2f}s\n")

        results.append({
            "image_id": item["image_id"],
            "file_name": item["file_name"],
            "ground_truth": item["ground_truth_captions"][0],
            "all_ground_truths": item["ground_truth_captions"],
            "baseline_greedy": caption_greedy[:200],
            "baseline_sampled": caption_sampled[:200],
            "baseline_creative": caption_creative[:200],
            "generation_time_sec": round(elapsed, 2),
        })

    # ─────────────────────────────────────────────────────────────────────
    # 5. DISPLAY & SAVE RESULTS
    # ─────────────────────────────────────────────────────────────────────

    df = pd.DataFrame(results)

    print("=" * 70)
    print("📊 BASELINE RESULTS SUMMARY")
    print("=" * 70)
    for _, row in df.iterrows():
        print(f"\n  Image: {row['file_name']}")
        print(f"  Truth:   {row['ground_truth']}")
        print(f"  Greedy:  {row['baseline_greedy'][:80]}")
        print(f"  Sampled: {row['baseline_sampled'][:80]}")

    # Save CSV
    csv_path = "/content/drive/MyDrive/image-captioning/baseline_captions.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n💾 CSV saved: {csv_path}")

    # Save JSON with full metadata
    json_path = "/content/drive/MyDrive/image-captioning/baseline_captions.json"
    record = {
        "experiment": "baseline_captioning",
        "model": "CLIPGpt2Prefix (untrained bridge)",
        "timestamp": datetime.now().isoformat(),
        "device": device,
        "config": {
            "num_prefix_tokens": 8,
            "clip_model": "ViT-B-32 (OpenCLIP)",
            "gpt2_model": "gpt2",
            "clip_frozen": True,
            "gpt2_frozen": False,
            "dataset": "COCO Captions train2017",
            "num_samples": len(results),
        },
        "results": results,
    }
    with open(json_path, "w") as f:
        json.dump(record, f, indent=2)
    print(f"💾 JSON saved: {json_path}")

    # ─────────────────────────────────────────────────────────────────────
    # 6. VISUALIZE — show images alongside captions
    # ─────────────────────────────────────────────────────────────────────

    import matplotlib.pyplot as plt
    import re

    def clean_caption(text, max_len=45):
        """Remove newlines, extra spaces, and truncate for display."""
        text = re.sub(r'\s+', ' ', text.strip())  # collapse all whitespace
        text = text.replace('"', "'")               # avoid quote rendering issues
        if len(text) == 0:
            text = "[empty — untrained]"
        if len(text) > max_len:
            text = text[:max_len] + "..."
        return text

    n = len(results)
    cols = min(5, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 5 * rows))
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for idx in range(len(axes)):
        ax = axes[idx]
        if idx < n:
            item = results[idx]
            img = Image.open(sample_data[idx]["path"]).convert("RGB")
            ax.imshow(img)

            gt_clean = clean_caption(item["ground_truth"])
            gen_clean = clean_caption(item["baseline_greedy"])

            ax.set_title(f"GT: {gt_clean}", fontsize=8, color="green", pad=8)
            ax.text(
                0.5, -0.08, f"Gen: {gen_clean}",
                transform=ax.transAxes, fontsize=8, color="red",
                ha="center", va="top", wrap=True
            )
        ax.set_xticks([])
        ax.set_yticks([])
        if idx >= n:
            ax.axis("off")

    fig.suptitle(
        "BASELINE: Green = Ground Truth | Red = Generated (untrained)",
        fontsize=14, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    plt.savefig(
        "/content/drive/MyDrive/image-captioning/baseline_visual.png",
        dpi=150, bbox_inches="tight"
    )
    plt.show()
    print("💾 Visualization saved: baseline_visual.png")

    print("\n" + "=" * 70)
    print("⚠️  Captions are EXPECTED to be nonsensical!")
    print("   The projection layer has random weights (untrained).")
    print("   After fine-tuning, re-run this to compare improvement.")
    print("=" * 70)

    return df, record


# ─────────────────────────────────────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    df, record = run_baseline()

In [ ]:
"""
═══════════════════════════════════════════════════════════════════════
MILESTONE 2: Train CLIP-GPT2 Prefix Model on COCO Captions
═══════════════════════════════════════════════════════════════════════
This loads your Milestone 1 processed data (embeddings + tokens)
and trains the projection layer so the model learns to generate
real captions from images.

Run cells in order in your Colab notebook.
"""

# ═════════════════════════════════════════════════════════════════════
# CELL 1: Setup & Load Milestone 1 Data
# ═════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import json
import time
import random
import numpy as np

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Load Milestone 1 saved data ──
SAVE_DIR = Path("/content/drive/MyDrive/image-captioning/processed")

print("Loading Milestone 1 data...")
data = torch.load(SAVE_DIR / "milestone1_tensors.pt", weights_only=True)
tokenizer = GPT2Tokenizer.from_pretrained(str(SAVE_DIR / "tokenizer"))

with open(SAVE_DIR / "milestone1_metadata.json", "r") as f:
    metadata = json.load(f)

image_embeddings = data["image_embeddings"]          # (5000, 512)
caption_input_ids = data["caption_input_ids"]        # (5000, 64)
caption_attention_masks = data["caption_attention_masks"]  # (5000, 64)
pairs = metadata["pairs"]                            # list of {image_id, path, caption}

print(f"✅ Loaded {len(pairs)} image-caption pairs")
print(f"   Embeddings: {image_embeddings.shape}")
print(f"   Tokens:     {caption_input_ids.shape}")
print(f"   Vocab size: {len(tokenizer)}")


# ═════════════════════════════════════════════════════════════════════
# CELL 2: Dataset & DataLoader
# ═════════════════════════════════════════════════════════════════════

class CocoCaptionDataset(Dataset):
    """
    Dataset that returns precomputed CLIP embeddings + tokenized captions.
    No image loading during training — everything is already preprocessed!
    """
    def __init__(self, embeddings, input_ids, attention_masks):
        self.embeddings = embeddings
        self.input_ids = input_ids
        self.attention_masks = attention_masks

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return {
            "clip_embedding": self.embeddings[idx],       # (512,)
            "input_ids": self.input_ids[idx],             # (64,)
            "attention_mask": self.attention_masks[idx],   # (64,)
        }


# Train/val split (90/10)
N = len(image_embeddings)
indices = list(range(N))
random.shuffle(indices)
split = int(0.9 * N)

train_idx = indices[:split]
val_idx = indices[split:]

train_dataset = CocoCaptionDataset(
    image_embeddings[train_idx],
    caption_input_ids[train_idx],
    caption_attention_masks[train_idx],
)
val_dataset = CocoCaptionDataset(
    image_embeddings[val_idx],
    caption_input_ids[val_idx],
    caption_attention_masks[val_idx],
)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_dataset)} samples ({len(val_loader)} batches)")


# ═════════════════════════════════════════════════════════════════════
# CELL 3: Model Definition
# ═════════════════════════════════════════════════════════════════════

class PrefixProjection(nn.Module):
    """
    Projects a CLIP embedding (512-dim) into K prefix tokens (each 768-dim)
    that GPT-2 can understand.

    512 → Linear → GELU → Linear → LayerNorm → reshape to (K, 768)
    """
    def __init__(self, clip_dim=512, gpt_dim=768, num_prefix_tokens=8):
        super().__init__()
        self.num_prefix_tokens = num_prefix_tokens
        self.gpt_dim = gpt_dim

        hidden_dim = gpt_dim * 2  # wider hidden layer for more capacity

        self.proj = nn.Sequential(
            nn.Linear(clip_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, gpt_dim * num_prefix_tokens),
            nn.LayerNorm(gpt_dim * num_prefix_tokens),
        )

    def forward(self, clip_emb):
        """(B, 512) → (B, K, 768)"""
        out = self.proj(clip_emb)
        return out.view(-1, self.num_prefix_tokens, self.gpt_dim)


class CLIPGpt2CaptionModel(nn.Module):
    """
    CLIP embedding → Prefix Projection → GPT-2 Language Model

    Uses PRECOMPUTED CLIP embeddings (no CLIP encoder needed during training).
    Only the prefix projection is trained. GPT-2 is partially unfrozen
    (last 2 layers + LM head) for better caption quality.
    """
    def __init__(self, num_prefix_tokens=8, unfreeze_last_n=2):
        super().__init__()

        # GPT-2
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt2.resize_token_embeddings(len(tokenizer))  # resize for special tokens

        gpt_dim = self.gpt2.config.n_embd  # 768

        # Prefix projection (the bridge we're training)
        self.prefix_proj = PrefixProjection(
            clip_dim=512, gpt_dim=gpt_dim, num_prefix_tokens=num_prefix_tokens
        )
        self.num_prefix = num_prefix_tokens

        # Freeze most of GPT-2, unfreeze last N layers + LM head
        for param in self.gpt2.parameters():
            param.requires_grad = False

        # Unfreeze last N transformer blocks
        for i in range(self.gpt2.config.n_layer - unfreeze_last_n, self.gpt2.config.n_layer):
            for param in self.gpt2.transformer.h[i].parameters():
                param.requires_grad = True

        # Unfreeze LM head and final layer norm
        for param in self.gpt2.lm_head.parameters():
            param.requires_grad = True
        for param in self.gpt2.transformer.ln_f.parameters():
            param.requires_grad = True

        # Unfreeze token embeddings (needed for new special tokens)
        for param in self.gpt2.transformer.wte.parameters():
            param.requires_grad = True

        # Count parameters
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Model built:")
        print(f"  Prefix tokens: {num_prefix_tokens}")
        print(f"  GPT-2 unfrozen layers: last {unfreeze_last_n}")
        print(f"  Total params:     {total/1e6:.1f}M")
        print(f"  Trainable params: {trainable/1e6:.1f}M ({100*trainable/total:.1f}%)")

    def forward(self, clip_embedding, input_ids, attention_mask):
        """
        Training forward pass.

        Args:
            clip_embedding: (B, 512) precomputed CLIP image embeddings
            input_ids:      (B, T) tokenized captions
            attention_mask:  (B, T) attention mask for captions
        Returns:
            loss: cross-entropy language modeling loss
            logits: (B, K+T, vocab_size)
        """
        B, T = input_ids.shape

        # 1. Project CLIP embedding → prefix tokens
        prefix_embeds = self.prefix_proj(clip_embedding)  # (B, K, 768)

        # 2. Get text token embeddings
        text_embeds = self.gpt2.transformer.wte(input_ids)  # (B, T, 768)

        # 3. Concatenate: [prefix | text]
        inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)  # (B, K+T, 768)

        # 4. Extend attention mask for prefix tokens (always attended to)
        prefix_mask = torch.ones(B, self.num_prefix, device=input_ids.device)
        full_mask = torch.cat([prefix_mask, attention_mask.float()], dim=1)  # (B, K+T)

        # 5. Create labels: -100 for prefix positions (no loss), input_ids for text
        prefix_labels = torch.full((B, self.num_prefix), -100, dtype=torch.long, device=input_ids.device)
        # For language modeling, labels = input_ids (shifted internally by HF)
        # Mask padding tokens with -100
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        full_labels = torch.cat([prefix_labels, labels], dim=1)  # (B, K+T)

        # 6. Forward through GPT-2
        outputs = self.gpt2(
            inputs_embeds=inputs_embeds,
            attention_mask=full_mask,
            labels=full_labels,
        )

        return outputs.loss, outputs.logits

    @torch.no_grad()
    def generate(self, clip_embedding, max_new_tokens=50, temperature=0.8, top_p=0.9):
        """
        Generate a caption from a CLIP embedding.

        Args:
            clip_embedding: (1, 512) or (B, 512) CLIP image embedding
            max_new_tokens: max words to generate
            temperature: sampling temperature (lower = more confident)
            top_p: nucleus sampling threshold
        Returns:
            list of generated caption strings
        """
        self.eval()
        if clip_embedding.dim() == 1:
            clip_embedding = clip_embedding.unsqueeze(0)

        B = clip_embedding.shape[0]
        device = clip_embedding.device

        # Get prefix
        prefix_embeds = self.prefix_proj(clip_embedding)  # (B, K, 768)

        # Start with BOS token
        cur_ids = torch.full((B, 1), tokenizer.bos_token_id, device=device)

        for _ in range(max_new_tokens):
            text_embeds = self.gpt2.transformer.wte(cur_ids)
            embeds = torch.cat([prefix_embeds, text_embeds], dim=1)

            outputs = self.gpt2(inputs_embeds=embeds)
            next_logits = outputs.logits[:, -1, :] / temperature

            # Nucleus (top-p) sampling
            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
            cumulative = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cumulative - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = -float("inf")
            probs = F.softmax(sorted_logits, dim=-1)
            sampled = torch.multinomial(probs, 1)
            next_token = sorted_idx.gather(-1, sampled)

            cur_ids = torch.cat([cur_ids, next_token], dim=1)

            if (next_token == tokenizer.eos_token_id).all():
                break

        captions = tokenizer.batch_decode(cur_ids, skip_special_tokens=True)
        return [c.strip() for c in captions]


# Build the model
NUM_PREFIX_TOKENS = 8
model = CLIPGpt2CaptionModel(num_prefix_tokens=NUM_PREFIX_TOKENS, unfreeze_last_n=2).to(DEVICE)


# ═════════════════════════════════════════════════════════════════════
# CELL 4: Training Loop
# ═════════════════════════════════════════════════════════════════════

# Hyperparameters
NUM_EPOCHS = 10
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100
LOG_EVERY = 25      # print loss every N steps
SAMPLE_EVERY = 1    # generate sample captions every N epochs

# Optimizer — only update trainable parameters
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Learning rate scheduler with warmup
total_steps = NUM_EPOCHS * len(train_loader)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS  # linear warmup
    return max(0.1, 1.0 - (step - WARMUP_STEPS) / (total_steps - WARMUP_STEPS))  # linear decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Mixed precision for faster training on T4
scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

# Training log
training_log = {
    "train_losses": [],
    "val_losses": [],
    "epoch_times": [],
    "learning_rates": [],
    "sample_captions": [],
}

# Pick 10 fixed images for tracking caption improvement
sample_indices = random.sample(range(len(val_dataset)), min(10, len(val_dataset)))
sample_embeddings = torch.stack([val_dataset[i]["clip_embedding"] for i in sample_indices]).to(DEVICE)
sample_ground_truths = [pairs[val_idx[i]]["caption"] for i in sample_indices]
sample_paths = [pairs[val_idx[i]]["path"] for i in sample_indices]

print(f"\nTraining config:")
print(f"  Epochs:        {NUM_EPOCHS}")
print(f"  Batch size:    {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Total steps:   {total_steps}")
print(f"  Warmup steps:  {WARMUP_STEPS}")
print(f"  Tracking {len(sample_indices)} sample images for caption evolution")
print(f"\n{'='*70}")
print("Starting training...")
print(f"{'='*70}\n")


best_val_loss = float("inf")
step = 0

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    # ── Training ──
    model.train()
    epoch_train_loss = 0
    num_batches = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        clip_emb = batch["clip_embedding"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)

        optimizer.zero_grad()

        if scaler is not None:
            with torch.amp.autocast("cuda"):
                loss, _ = model(clip_emb, input_ids, attn_mask)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss, _ = model(clip_emb, input_ids, attn_mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()
        step += 1
        epoch_train_loss += loss.item()
        num_batches += 1

        if step % LOG_EVERY == 0:
            lr = scheduler.get_last_lr()[0]
            print(f"  Step {step:4d} | Loss: {loss.item():.4f} | LR: {lr:.2e}")

    avg_train_loss = epoch_train_loss / num_batches

    # ── Validation ──
    model.eval()
    epoch_val_loss = 0
    val_batches = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            clip_emb = batch["clip_embedding"].to(DEVICE)
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)

            if scaler is not None:
                with torch.amp.autocast("cuda"):
                    loss, _ = model(clip_emb, input_ids, attn_mask)
            else:
                loss, _ = model(clip_emb, input_ids, attn_mask)

            epoch_val_loss += loss.item()
            val_batches += 1

    avg_val_loss = epoch_val_loss / val_batches
    epoch_time = time.time() - epoch_start

    # Log
    training_log["train_losses"].append(avg_train_loss)
    training_log["val_losses"].append(avg_val_loss)
    training_log["epoch_times"].append(round(epoch_time, 1))
    training_log["learning_rates"].append(scheduler.get_last_lr()[0])

    print(f"\n{'─'*70}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} complete in {epoch_time:.0f}s")
    print(f"  Train Loss: {avg_train_loss:.4f}  |  Val Loss: {avg_val_loss:.4f}")
    print(f"{'─'*70}")

    # ── Save best model ──
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        save_path = Path("/content/drive/MyDrive/image-captioning/models")
        save_path.mkdir(parents=True, exist_ok=True)
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_loss": avg_val_loss,
            "config": {
                "num_prefix_tokens": NUM_PREFIX_TOKENS,
                "unfreeze_last_n": 2,
            },
        }, save_path / "best_model.pt")
        print(f"  💾 New best model saved (val_loss: {avg_val_loss:.4f})")

    # ── Generate sample captions every epoch ──
    if (epoch + 1) % SAMPLE_EVERY == 0:
        print(f"\n  📝 Sample captions (Epoch {epoch+1}):")
        captions = model.generate(sample_embeddings, max_new_tokens=40, temperature=0.7, top_p=0.9)

        epoch_samples = []
        for i in range(min(5, len(captions))):  # show first 5
            print(f"    GT:  {sample_ground_truths[i]}")
            print(f"    Gen: {captions[i]}")
            print()
            epoch_samples.append({
                "ground_truth": sample_ground_truths[i],
                "generated": captions[i],
                "image_path": sample_paths[i],
            })

        training_log["sample_captions"].append({
            "epoch": epoch + 1,
            "samples": epoch_samples,
        })

    print()

print(f"{'='*70}")
print(f"Training complete! Best val loss: {best_val_loss:.4f}")
print(f"{'='*70}")


# ═════════════════════════════════════════════════════════════════════
# CELL 5: Save Training Log & Plot Loss Curves
# ═════════════════════════════════════════════════════════════════════

# Save training log
log_path = Path("/content/drive/MyDrive/image-captioning/training_log.json")
with open(log_path, "w") as f:
    json.dump(training_log, f, indent=2)
print(f"💾 Training log saved: {log_path}")

# Plot loss curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

# Loss curve
ax1.plot(epochs_range, training_log["train_losses"], "b-o", label="Train Loss", markersize=4)
ax1.plot(epochs_range, training_log["val_losses"], "r-o", label="Val Loss", markersize=4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Learning rate curve
ax2.plot(epochs_range, training_log["learning_rates"], "g-o", markersize=4)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Learning Rate")
ax2.set_title("Learning Rate Schedule")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/image-captioning/loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Loss curves saved: loss_curves.png")


# ═════════════════════════════════════════════════════════════════════
# CELL 6: Generate Final Caption Samples (10 images)
# ═════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print("📝 FINAL CAPTION SAMPLES (after training)")
print(f"{'='*70}\n")

# Load best model
best_checkpoint = torch.load(
    "/content/drive/MyDrive/image-captioning/models/best_model.pt",
    weights_only=True,
)
model.load_state_dict(best_checkpoint["model_state_dict"])
model.eval()
print(f"Loaded best model from epoch {best_checkpoint['epoch']+1} (val_loss: {best_checkpoint['val_loss']:.4f})\n")

# Generate captions for all 10 tracked images
final_captions = model.generate(sample_embeddings, max_new_tokens=50, temperature=0.7, top_p=0.9)

final_results = []
for i in range(len(sample_indices)):
    print(f"── Image {i+1}/{len(sample_indices)}: {Path(sample_paths[i]).name} ──")
    print(f"   Ground Truth: {sample_ground_truths[i]}")
    print(f"   Generated:    {final_captions[i]}")
    print()
    final_results.append({
        "image": Path(sample_paths[i]).name,
        "ground_truth": sample_ground_truths[i],
        "generated": final_captions[i],
    })

# Save final results
final_path = Path("/content/drive/MyDrive/image-captioning/final_caption_samples.json")
with open(final_path, "w") as f:
    json.dump(final_results, f, indent=2)
print(f"💾 Final samples saved: {final_path}")


# ═════════════════════════════════════════════════════════════════════
# CELL 7: Visual Comparison Grid (GT vs Generated)
# ═════════════════════════════════════════════════════════════════════

import re

def clean_text(text, max_len=50):
    text = re.sub(r'\s+', ' ', str(text).strip())
    if not text:
        return "[empty]"
    return text[:max_len] + "..." if len(text) > max_len else text

n = len(final_results)
cols = min(5, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 5.5 * rows))
if rows == 1:
    axes = [axes] if cols == 1 else list(axes)
else:
    axes = axes.flatten()

for idx in range(len(axes)):
    ax = axes[idx]
    if idx < n:
        try:
            img = Image.open(sample_paths[idx]).convert("RGB")
            ax.imshow(img)
        except:
            ax.text(0.5, 0.5, "Image\nnot found", ha="center", va="center", transform=ax.transAxes)

        gt = clean_text(final_results[idx]["ground_truth"])
        gen = clean_text(final_results[idx]["generated"])

        ax.set_title(f"GT: {gt}", fontsize=8, color="green", pad=8)
        ax.text(0.5, -0.08, f"Gen: {gen}", transform=ax.transAxes,
                fontsize=8, color="blue", ha="center", va="top")
    ax.set_xticks([])
    ax.set_yticks([])
    if idx >= n:
        ax.axis("off")

fig.suptitle("AFTER TRAINING: Green = Ground Truth | Blue = Generated",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/image-captioning/trained_caption_samples.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("💾 Visual comparison saved: trained_caption_samples.png")

print(f"\n{'='*70}")
print("✅ MILESTONE 2 TRAINING COMPLETE!")
print(f"{'='*70}")
print(f"""
Files saved to Google Drive:
  📁 /image-captioning/
     ├── models/best_model.pt          (trained model weights)
     ├── training_log.json             (loss per epoch + sample captions)
     ├── loss_curves.png               (train/val loss plot)
     ├── final_caption_samples.json    (10 images with GT + generated captions)
     └── trained_caption_samples.png   (visual grid: GT vs generated)
""")

In [ ]:
"""
Decoding Strategies Experiment
Compare: Greedy, Beam Search, Top-K Sampling, Nucleus (Top-P) Sampling,
         and Temperature variations.

Uses your existing CLIPGpt2Prefix model + COCO images from Milestone 1.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from PIL import Image
from pathlib import Path
from pycocotools.coco import COCO
import pandas as pd
import random
import time
import json
from datetime import datetime
from typing import Optional


# ═════════════════════════════════════════════════════════════════════════════
# 1. MODEL (same CLIPGpt2Prefix but with ALL decoding strategies built in)
# ═════════════════════════════════════════════════════════════════════════════

class CLIPVisualEncoder(nn.Module):
    def __init__(self, model_name="ViT-B-32", pretrained="openai", freeze=True):
        super().__init__()
        clip_model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name, pretrained=pretrained
        )
        self.visual = clip_model.visual
        self.embed_dim = clip_model.visual.output_dim
        if freeze:
            for p in self.visual.parameters():
                p.requires_grad = False

    @torch.no_grad()
    def forward(self, pixel_values):
        x = self.visual.conv1(pixel_values)
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        x = torch.cat([
            self.visual.class_embedding.unsqueeze(0).expand(x.shape[0], -1, -1), x
        ], dim=1)
        x = x + self.visual.positional_embedding.unsqueeze(0)
        x = self.visual.ln_pre(x)
        x = self.visual.transformer(x)
        patch_tokens = self.visual.ln_post(x[:, 1:, :])
        cls_token = x[:, 0, :] @ self.visual.proj
        return cls_token, patch_tokens


class LinearProjection(nn.Module):
    def __init__(self, clip_dim, gpt_dim, num_prefix_tokens=8):
        super().__init__()
        self.num_prefix_tokens = num_prefix_tokens
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, gpt_dim * num_prefix_tokens),
            nn.GELU(),
            nn.LayerNorm(gpt_dim * num_prefix_tokens),
        )
        self.gpt_dim = gpt_dim

    def forward(self, cls_token):
        out = self.proj(cls_token)
        return out.view(-1, self.num_prefix_tokens, self.gpt_dim)


class CLIPGpt2WithDecodingStrategies(nn.Module):
    """
    CLIPGpt2Prefix model with 5 different decoding strategies:
      1. Greedy           — always pick the highest probability token
      2. Beam Search      — track top-B sequences in parallel
      3. Top-K Sampling   — sample from the top K most likely tokens
      4. Nucleus (Top-P)  — sample from smallest set with cumulative prob >= P
      5. Temperature      — scale logits to control randomness
    """

    def __init__(self, num_prefix_tokens=8):
        super().__init__()
        self.clip = CLIPVisualEncoder(freeze=True)
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        self.tokenizer.pad_token = self.tokenizer.eos_token

        gpt_dim = self.gpt2.config.n_embd
        clip_dim = self.clip.embed_dim
        self.prefix_proj = LinearProjection(clip_dim, gpt_dim, num_prefix_tokens)
        self.num_prefix = num_prefix_tokens

    def _get_prefix_and_start(self, pixel_values):
        """Encode image and prepare prefix embeddings + start token."""
        cls_token, _ = self.clip(pixel_values)
        prefix_embeds = self.prefix_proj(cls_token)
        B = pixel_values.shape[0]
        device = pixel_values.device
        start_ids = torch.full((B, 1), self.tokenizer.bos_token_id, device=device)
        return prefix_embeds, start_ids

    def _forward_with_prefix(self, prefix_embeds, token_ids):
        """Run GPT-2 with visual prefix + token sequence."""
        text_embeds = self.gpt2.transformer.wte(token_ids)
        embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
        outputs = self.gpt2(inputs_embeds=embeds)
        return outputs.logits[:, -1, :]  # logits for next token

    # ─────────────────────────────────────────────────────────────────────
    # STRATEGY 1: GREEDY DECODING
    # ─────────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def generate_greedy(self, pixel_values, max_new_tokens=40):
        """
        Always pick the single most probable next token.

        Pros: Deterministic, fast, no randomness
        Cons: Repetitive, boring, gets stuck in loops
        """
        self.eval()
        prefix_embeds, generated = self._get_prefix_and_start(pixel_values)

        for _ in range(max_new_tokens):
            logits = self._forward_with_prefix(prefix_embeds, generated)
            next_token = logits.argmax(dim=-1, keepdim=True)  # pick highest prob
            generated = torch.cat([generated, next_token], dim=1)
            if (next_token == self.tokenizer.eos_token_id).all():
                break

        return self.tokenizer.batch_decode(generated, skip_special_tokens=True)

    # ─────────────────────────────────────────────────────────────────────
    # STRATEGY 2: BEAM SEARCH
    # ─────────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def generate_beam_search(self, pixel_values, max_new_tokens=40, num_beams=5):
        """
        Keep top-B candidate sequences at each step, expand all of them,
        then keep the best B again. Returns the highest-scoring complete sequence.

        Pros: Finds globally better sequences than greedy
        Cons: Slower (B times more compute), still deterministic, can be repetitive
        """
        self.eval()
        device = pixel_values.device
        cls_token, _ = self.clip(pixel_values)
        prefix_embeds = self.prefix_proj(cls_token)  # [1, K, D]

        # Initialize beams: (sequence, cumulative_log_prob)
        start_id = self.tokenizer.bos_token_id
        beams = [(torch.tensor([[start_id]], device=device), 0.0)]
        completed = []

        for step in range(max_new_tokens):
            candidates = []

            for seq, score in beams:
                # Get next token probabilities
                text_embeds = self.gpt2.transformer.wte(seq)
                embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
                outputs = self.gpt2(inputs_embeds=embeds)
                logits = outputs.logits[:, -1, :]
                log_probs = F.log_softmax(logits, dim=-1)

                # Get top-B tokens for this beam
                top_log_probs, top_ids = log_probs.topk(num_beams, dim=-1)

                for i in range(num_beams):
                    token_id = top_ids[0, i].item()
                    new_score = score + top_log_probs[0, i].item()
                    new_seq = torch.cat([
                        seq, torch.tensor([[token_id]], device=device)
                    ], dim=1)

                    if token_id == self.tokenizer.eos_token_id:
                        completed.append((new_seq, new_score))
                    else:
                        candidates.append((new_seq, new_score))

            if not candidates:
                break

            # Keep only top-B candidates
            candidates.sort(key=lambda x: x[1], reverse=True)
            beams = candidates[:num_beams]

        # Pick best from completed (or best beam if none completed)
        all_options = completed + beams
        all_options.sort(key=lambda x: x[1] / x[0].size(1), reverse=True)  # length-normalized
        best_seq = all_options[0][0]

        return self.tokenizer.batch_decode(best_seq, skip_special_tokens=True)

    # ─────────────────────────────────────────────────────────────────────
    # STRATEGY 3: TOP-K SAMPLING
    # ─────────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def generate_top_k(self, pixel_values, max_new_tokens=40, k=50, temperature=1.0):
        """
        At each step, zero out everything except the top K most probable tokens,
        then sample from those K tokens.

        Pros: More diverse than greedy, avoids very unlikely tokens
        Cons: Fixed K doesn't adapt to the probability distribution
              (sometimes 10 tokens make sense, sometimes 500 do)
        """
        self.eval()
        prefix_embeds, generated = self._get_prefix_and_start(pixel_values)

        for _ in range(max_new_tokens):
            logits = self._forward_with_prefix(prefix_embeds, generated) / temperature

            # Keep only top-K, set rest to -inf
            top_k_vals, top_k_idx = logits.topk(k, dim=-1)
            filtered = torch.full_like(logits, -float("inf"))
            filtered.scatter_(-1, top_k_idx, top_k_vals)

            probs = F.softmax(filtered, dim=-1)
            next_token = torch.multinomial(probs, 1)
            generated = torch.cat([generated, next_token], dim=1)

            if (next_token == self.tokenizer.eos_token_id).all():
                break

        return self.tokenizer.batch_decode(generated, skip_special_tokens=True)

    # ─────────────────────────────────────────────────────────────────────
    # STRATEGY 4: NUCLEUS (TOP-P) SAMPLING
    # ─────────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def generate_nucleus(self, pixel_values, max_new_tokens=40, top_p=0.9, temperature=1.0):
        """
        Sort tokens by probability, keep the smallest set whose cumulative
        probability >= top_p, then sample from that set.

        Pros: Adapts to the distribution — uses fewer tokens when model is
              confident, more when uncertain. Best quality in practice.
        Cons: Slightly more compute than top-k
        """
        self.eval()
        prefix_embeds, generated = self._get_prefix_and_start(pixel_values)

        for _ in range(max_new_tokens):
            logits = self._forward_with_prefix(prefix_embeds, generated) / temperature

            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cumulative = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            # Remove tokens with cumulative prob above top_p
            mask = cumulative - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = -float("inf")

            probs = F.softmax(sorted_logits, dim=-1)
            sampled = torch.multinomial(probs, 1)
            next_token = sorted_idx.gather(-1, sampled)

            generated = torch.cat([generated, next_token], dim=1)
            if (next_token == self.tokenizer.eos_token_id).all():
                break

        return self.tokenizer.batch_decode(generated, skip_special_tokens=True)

    # ─────────────────────────────────────────────────────────────────────
    # STRATEGY 5: TEMPERATURE SCALING (with nucleus)
    # ─────────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def generate_temperature(self, pixel_values, max_new_tokens=40, temperature=0.5):
        """
        Scale logits by temperature before sampling:
          - temperature < 1.0 → sharper distribution → more confident/repetitive
          - temperature = 1.0 → unchanged
          - temperature > 1.0 → flatter distribution → more random/creative

        Combined with nucleus sampling (top_p=0.9) for best results.
        """
        return self.generate_nucleus(
            pixel_values, max_new_tokens=max_new_tokens,
            top_p=0.9, temperature=temperature
        )


# ═════════════════════════════════════════════════════════════════════════════
# 2. LOAD YOUR COCO DATA
# ═════════════════════════════════════════════════════════════════════════════

DATA_DIR = Path("/content/drive/MyDrive/image-captioning/data")
IMG_DIR  = DATA_DIR / "train2017"
ANN_FILE = DATA_DIR / "annotations" / "captions_train2017.json"

assert IMG_DIR.exists(),  f"❌ Images not found: {IMG_DIR}"
assert ANN_FILE.exists(), f"❌ Annotations not found: {ANN_FILE}"

SEED = 42
NUM_SAMPLES = 5
random.seed(SEED)

coco = COCO(str(ANN_FILE))
all_img_ids = list(coco.imgs.keys())

# Only pick from images that exist on disk
available_ids = [
    img_id for img_id in all_img_ids
    if (IMG_DIR / coco.imgs[img_id]["file_name"]).exists()
]
print(f"Available images on disk: {len(available_ids)}")

sample_ids = random.sample(available_ids, NUM_SAMPLES)

sample_data = []
for img_id in sample_ids:
    info = coco.imgs[img_id]
    fname = info["file_name"]
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)
    captions = [a["caption"].strip() for a in anns]
    sample_data.append({
        "image_id": img_id,
        "file_name": fname,
        "path": str(IMG_DIR / fname),
        "ground_truth": captions[0],
        "all_ground_truths": captions,
    })

print(f"Selected {len(sample_data)} images for decoding experiment\n")


# ═════════════════════════════════════════════════════════════════════════════
# 3. RUN ALL DECODING STRATEGIES
# ═════════════════════════════════════════════════════════════════════════════

def run_decoding_experiment():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("📦 Building model...")
    model = CLIPGpt2WithDecodingStrategies(num_prefix_tokens=8).to(device)
    print("✅ Model ready\n")

    # Define all strategies to test
    strategies = {
        "greedy": {
            "fn": model.generate_greedy,
            "kwargs": {},
            "description": "Always picks highest-probability token",
        },
        "beam_search_3": {
            "fn": model.generate_beam_search,
            "kwargs": {"num_beams": 3},
            "description": "Beam search with 3 beams",
        },
        "beam_search_5": {
            "fn": model.generate_beam_search,
            "kwargs": {"num_beams": 5},
            "description": "Beam search with 5 beams",
        },
        "top_k_10": {
            "fn": model.generate_top_k,
            "kwargs": {"k": 10, "temperature": 1.0},
            "description": "Sample from top 10 tokens",
        },
        "top_k_50": {
            "fn": model.generate_top_k,
            "kwargs": {"k": 50, "temperature": 1.0},
            "description": "Sample from top 50 tokens",
        },
        "nucleus_p09": {
            "fn": model.generate_nucleus,
            "kwargs": {"top_p": 0.9, "temperature": 1.0},
            "description": "Nucleus sampling with p=0.9",
        },
        "nucleus_p05": {
            "fn": model.generate_nucleus,
            "kwargs": {"top_p": 0.5, "temperature": 1.0},
            "description": "Nucleus sampling with p=0.5 (more focused)",
        },
        "temp_0.3": {
            "fn": model.generate_temperature,
            "kwargs": {"temperature": 0.3},
            "description": "Low temperature — very confident/repetitive",
        },
        "temp_0.7": {
            "fn": model.generate_temperature,
            "kwargs": {"temperature": 0.7},
            "description": "Medium temperature — balanced",
        },
        "temp_1.5": {
            "fn": model.generate_temperature,
            "kwargs": {"temperature": 1.5},
            "description": "High temperature — creative/random",
        },
    }

    # Run each strategy on each image
    all_results = []

    for img_idx, item in enumerate(sample_data):
        print(f"{'='*70}")
        print(f"🖼️  [{img_idx+1}/{len(sample_data)}] {item['file_name']}")
        print(f"   Ground Truth: {item['ground_truth']}")
        print(f"{'='*70}")

        img = Image.open(item["path"]).convert("RGB")
        pixel_values = model.clip.preprocess(img).unsqueeze(0).to(device)

        image_results = {
            "image_id": item["image_id"],
            "file_name": item["file_name"],
            "ground_truth": item["ground_truth"],
        }

        for name, config in strategies.items():
            start = time.time()
            caption = config["fn"](pixel_values, max_new_tokens=40, **config["kwargs"])
            elapsed = time.time() - start

            # Handle list output
            cap_text = caption[0] if isinstance(caption, list) else caption
            cap_text = " ".join(cap_text.split())[:150]  # clean whitespace

            image_results[f"{name}_caption"] = cap_text
            image_results[f"{name}_time"] = round(elapsed, 3)

            print(f"   {name:20s} ({elapsed:.2f}s): {cap_text[:80]}")

        print()
        all_results.append(image_results)

    return all_results, strategies


# ═════════════════════════════════════════════════════════════════════════════
# 4. ANALYZE & COMPARE RESULTS
# ═════════════════════════════════════════════════════════════════════════════

def analyze_results(all_results, strategies):
    import matplotlib.pyplot as plt
    import re

    def clean(text, max_len=50):
        text = re.sub(r'\s+', ' ', str(text).strip())
        return text[:max_len] + "..." if len(text) > max_len else text if text else "[empty]"

    strategy_names = list(strategies.keys())

    # ── Table: all captions side by side ──
    print("\n" + "=" * 70)
    print("📊 FULL COMPARISON TABLE")
    print("=" * 70)

    for result in all_results:
        print(f"\n🖼️  {result['file_name']}")
        print(f"   {'Ground Truth':20s}: {clean(result['ground_truth'], 70)}")
        print(f"   {'─'*60}")
        for name in strategy_names:
            cap = result.get(f"{name}_caption", "")
            t = result.get(f"{name}_time", 0)
            print(f"   {name:20s}: {clean(cap, 60)}  ({t:.2f}s)")

    # ── Timing comparison ──
    print("\n" + "=" * 70)
    print("⏱️  AVERAGE GENERATION TIME PER STRATEGY")
    print("=" * 70)

    avg_times = {}
    for name in strategy_names:
        times = [r.get(f"{name}_time", 0) for r in all_results]
        avg_times[name] = sum(times) / len(times)
        print(f"   {name:20s}: {avg_times[name]:.3f}s avg")

    # ── Caption length comparison ──
    print("\n" + "=" * 70)
    print("📏 AVERAGE CAPTION LENGTH (words)")
    print("=" * 70)

    avg_lengths = {}
    for name in strategy_names:
        lengths = [len(r.get(f"{name}_caption", "").split()) for r in all_results]
        avg_lengths[name] = sum(lengths) / len(lengths)
        print(f"   {name:20s}: {avg_lengths[name]:.1f} words avg")

    # ── Visualization: Timing bar chart ──
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    colors = plt.cm.Set3([i / len(strategy_names) for i in range(len(strategy_names))])

    # Time chart
    bars1 = ax1.barh(strategy_names, [avg_times[n] for n in strategy_names], color=colors)
    ax1.set_xlabel("Average Time (seconds)")
    ax1.set_title("Generation Speed by Strategy")
    ax1.invert_yaxis()
    for bar, name in zip(bars1, strategy_names):
        ax1.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{avg_times[name]:.3f}s", va="center", fontsize=8)

    # Length chart
    bars2 = ax2.barh(strategy_names, [avg_lengths[n] for n in strategy_names], color=colors)
    ax2.set_xlabel("Average Words")
    ax2.set_title("Caption Length by Strategy")
    ax2.invert_yaxis()
    for bar, name in zip(bars2, strategy_names):
        ax2.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f"{avg_lengths[name]:.1f}", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/image-captioning/decoding_comparison.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    # ── Visualization: Per-image comparison grid ──
    fig, axes = plt.subplots(len(all_results), 1, figsize=(14, 5 * len(all_results)))
    if len(all_results) == 1:
        axes = [axes]

    for idx, (ax, result) in enumerate(zip(axes, all_results)):
        img = Image.open(sample_data[idx]["path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"GT: {clean(result['ground_truth'], 60)}", fontsize=10,
                     color="green", fontweight="bold")

        # Build caption text block
        caption_lines = []
        for name in strategy_names:
            cap = clean(result.get(f"{name}_caption", ""), 50)
            caption_lines.append(f"{name}: {cap}")
        caption_block = "\n".join(caption_lines)

        ax.text(1.02, 0.5, caption_block, transform=ax.transAxes,
                fontsize=7, fontfamily="monospace", va="center",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.9))
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle("Decoding Strategies Comparison (Untrained Model)",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/image-captioning/decoding_per_image.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    # ── Save everything ──
    save_path = "/content/drive/MyDrive/image-captioning/decoding_experiment.json"
    record = {
        "experiment": "decoding_strategies_comparison",
        "timestamp": datetime.now().isoformat(),
        "model": "CLIPGpt2Prefix (untrained bridge)",
        "strategies": {k: v["description"] for k, v in strategies.items()},
        "avg_times": avg_times,
        "avg_lengths": avg_lengths,
        "results": all_results,
    }
    with open(save_path, "w") as f:
        json.dump(record, f, indent=2)
    print(f"\n💾 Full results saved: {save_path}")
    print("💾 Charts saved: decoding_comparison.png, decoding_per_image.png")

    return avg_times, avg_lengths


# ═════════════════════════════════════════════════════════════════════════════
# 5. EXPLAIN WHAT EACH STRATEGY DOES
# ═════════════════════════════════════════════════════════════════════════════

def print_strategy_explainer():
    print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    DECODING STRATEGIES EXPLAINED                     ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. GREEDY                                                           ║
║     At each step, pick the single most probable token.               ║
║     Fast but gets stuck in repetitive loops like "The The The..."    ║
║                                                                      ║
║  2. BEAM SEARCH                                                      ║
║     Keep B best partial sequences at each step. Expand all, prune.   ║
║     Finds better overall sequences but slower (B× compute).         ║
║     Still deterministic — same input always gives same output.       ║
║                                                                      ║
║  3. TOP-K SAMPLING                                                   ║
║     Only consider the K most likely tokens, then randomly sample.    ║
║     K=10: focused, fewer surprises                                   ║
║     K=50: more diverse, occasional surprises                         ║
║     Problem: K is fixed regardless of how confident the model is.    ║
║                                                                      ║
║  4. NUCLEUS (TOP-P) SAMPLING  ← generally the best                  ║
║     Sort tokens by probability. Keep the smallest set whose          ║
║     cumulative probability >= P. Sample from that set.               ║
║     P=0.5: focused (keeps only the "core" tokens)                    ║
║     P=0.9: diverse (keeps most reasonable tokens)                    ║
║     Adapts to context — uses 3 tokens when confident, 100 when not. ║
║                                                                      ║
║  5. TEMPERATURE                                                      ║
║     Scale logits before sampling: logits / temperature               ║
║     T=0.3: sharp distribution → almost greedy, very repetitive      ║
║     T=0.7: slightly smoothed → balanced                             ║
║     T=1.0: unchanged                                                ║
║     T=1.5: flat distribution → random, creative, sometimes garbage  ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
    """)


# ═════════════════════════════════════════════════════════════════════════════
# RUN EVERYTHING
# ═════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print_strategy_explainer()
    all_results, strategies = run_decoding_experiment()
    avg_times, avg_lengths = analyze_results(all_results, strategies)

Milestone 3

In [ ]:
!pip install rouge-score pycocoevalcap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

In [ ]:
 !pip install nltk rouge-score pycocoevalcap matplotlib seaborn pycocotools

import os, json, random, time, warnings, re
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    CLIPModel, CLIPProcessor,
    GPT2LMHeadModel, GPT2Tokenizer,
)
from pycocotools.coco import COCO

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device: {DEVICE}")



In [ ]:
#=============================================================================
# LOAD YOUR TOKENIZER (with special tokens from Milestone 1)
# =============================================================================

SAVE_DIR = Path("/content/drive/MyDrive/image-captioning/processed")

# Mount Drive if not already mounted
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Load tokenizer from local path (not HuggingFace hub)
tok_path = str(SAVE_DIR / "tokenizer")
tokenizer = GPT2Tokenizer.from_pretrained(tok_path, local_files_only=True)
print(f"[INFO] Tokenizer loaded — vocab size: {len(tokenizer)}")
print(f"  BOS: '{tokenizer.bos_token}' (ID {tokenizer.bos_token_id})")
print(f"  EOS: '{tokenizer.eos_token}' (ID {tokenizer.eos_token_id})")
print(f"  PAD: '{tokenizer.pad_token}' (ID {tokenizer.pad_token_id})")


In [ ]:
 #=============================================================================
# Cell 3: MODEL DEFINITION (exactly as your Milestone 2 training code)
# =============================================================================

class PrefixProjection(nn.Module):
    """512 → Linear → GELU → Linear → LayerNorm → reshape to (K, 768)"""
    def __init__(self, clip_dim=512, gpt_dim=768, num_prefix_tokens=8):
        super().__init__()
        self.num_prefix_tokens = num_prefix_tokens
        self.gpt_dim = gpt_dim
        hidden_dim = gpt_dim * 2
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, gpt_dim * num_prefix_tokens),
            nn.LayerNorm(gpt_dim * num_prefix_tokens),
        )

    def forward(self, clip_emb):
        out = self.proj(clip_emb)
        return out.view(-1, self.num_prefix_tokens, self.gpt_dim)


class CLIPGpt2CaptionModel(nn.Module):
    """
    Same model from your Milestone 2 training code.
    Uses precomputed CLIP embeddings as input.
    """
    def __init__(self, num_prefix_tokens=8, unfreeze_last_n=2):
        super().__init__()
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt2.resize_token_embeddings(len(tokenizer))

        gpt_dim = self.gpt2.config.n_embd  # 768
        self.prefix_proj = PrefixProjection(
            clip_dim=512, gpt_dim=gpt_dim, num_prefix_tokens=num_prefix_tokens
        )
        self.num_prefix = num_prefix_tokens

        # Freeze most of GPT-2, unfreeze last N layers + LM head
        for param in self.gpt2.parameters():
            param.requires_grad = False
        for i in range(self.gpt2.config.n_layer - unfreeze_last_n, self.gpt2.config.n_layer):
            for param in self.gpt2.transformer.h[i].parameters():
                param.requires_grad = True
        for param in self.gpt2.lm_head.parameters():
            param.requires_grad = True
        for param in self.gpt2.transformer.ln_f.parameters():
            param.requires_grad = True
        for param in self.gpt2.transformer.wte.parameters():
            param.requires_grad = True

    @torch.no_grad()
    def generate(self, clip_embedding, max_new_tokens=50, temperature=0.8, top_p=0.9):
        """Generate caption from a CLIP embedding."""
        self.eval()
        if clip_embedding.dim() == 1:
            clip_embedding = clip_embedding.unsqueeze(0)

        B = clip_embedding.shape[0]
        device = clip_embedding.device
        prefix_embeds = self.prefix_proj(clip_embedding)
        cur_ids = torch.full((B, 1), tokenizer.bos_token_id, device=device)

        for _ in range(max_new_tokens):
            text_embeds = self.gpt2.transformer.wte(cur_ids)
            embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
            outputs = self.gpt2(inputs_embeds=embeds)
            next_logits = outputs.logits[:, -1, :] / temperature

            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
            cumulative = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cumulative - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = -float("inf")
            probs = F.softmax(sorted_logits, dim=-1)
            sampled = torch.multinomial(probs, 1)
            next_token = sorted_idx.gather(-1, sampled)

            cur_ids = torch.cat([cur_ids, next_token], dim=1)
            if (next_token == tokenizer.eos_token_id).all():
                break

        captions = tokenizer.batch_decode(cur_ids, skip_special_tokens=True)
        return [c.strip() for c in captions]

    @torch.no_grad()
    def generate_greedy(self, clip_embedding, max_new_tokens=50):
        """Greedy decoding — always pick highest probability token."""
        self.eval()
        if clip_embedding.dim() == 1:
            clip_embedding = clip_embedding.unsqueeze(0)

        B = clip_embedding.shape[0]
        device = clip_embedding.device
        prefix_embeds = self.prefix_proj(clip_embedding)
        cur_ids = torch.full((B, 1), tokenizer.bos_token_id, device=device)

        for _ in range(max_new_tokens):
            text_embeds = self.gpt2.transformer.wte(cur_ids)
            embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
            outputs = self.gpt2(inputs_embeds=embeds)
            next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            cur_ids = torch.cat([cur_ids, next_token], dim=1)
            if (next_token == tokenizer.eos_token_id).all():
                break

        captions = tokenizer.batch_decode(cur_ids, skip_special_tokens=True)
        return [c.strip() for c in captions]

    @torch.no_grad()
    def generate_beam(self, clip_embedding, max_new_tokens=50, num_beams=5):
        """Beam search decoding."""
        self.eval()
        if clip_embedding.dim() == 1:
            clip_embedding = clip_embedding.unsqueeze(0)

        device = clip_embedding.device
        prefix_embeds = self.prefix_proj(clip_embedding)
        start_id = tokenizer.bos_token_id
        beams = [(torch.tensor([[start_id]], device=device), 0.0)]
        completed = []

        for _ in range(max_new_tokens):
            candidates = []
            for seq, score in beams:
                text_embeds = self.gpt2.transformer.wte(seq)
                embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
                outputs = self.gpt2(inputs_embeds=embeds)
                log_probs = F.log_softmax(outputs.logits[:, -1, :], dim=-1)
                top_lp, top_ids = log_probs.topk(num_beams, dim=-1)

                for i in range(num_beams):
                    tid = top_ids[0, i].item()
                    ns = score + top_lp[0, i].item()
                    nseq = torch.cat([seq, torch.tensor([[tid]], device=device)], dim=1)
                    if tid == tokenizer.eos_token_id:
                        completed.append((nseq, ns))
                    else:
                        candidates.append((nseq, ns))

            if not candidates:
                break
            candidates.sort(key=lambda x: x[1], reverse=True)
            beams = candidates[:num_beams]

        all_opts = completed + beams
        all_opts.sort(key=lambda x: x[1] / x[0].size(1), reverse=True)
        best = all_opts[0][0]
        return tokenizer.batch_decode(best, skip_special_tokens=True)



In [ ]:
#=============================================================================
# Cell 4: LOAD CLIP ENCODER + TRAINED CHECKPOINT
# =============================================================================

# ── Load CLIP for encoding raw images ──
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model.eval()
print(f"[INFO] CLIP loaded: {CLIP_MODEL_NAME}")

# ── Build caption model and load trained weights ──
model = CLIPGpt2CaptionModel(num_prefix_tokens=8, unfreeze_last_n=2).to(DEVICE)

CHECKPOINT_PATH = "/content/drive/MyDrive/image-captioning/models/best_model.pt"
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"[INFO] Loaded trained model from epoch {checkpoint['epoch'] + 1}")
print(f"[INFO] Val loss: {checkpoint['val_loss']:.4f}")


# ── Helper: encode raw image → CLIP embedding ──
@torch.no_grad()
def encode_image(image):
    """PIL Image → (1, 512) CLIP embedding."""
    inputs = clip_processor(images=image, return_tensors="pt").to(DEVICE)
    outputs = clip_model.get_image_features(**inputs)
    # Handle both tensor and object returns
    if isinstance(outputs, torch.Tensor):
        feats = outputs
    else:
        feats = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs[0]
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats


In [ ]:
 #=============================================================================
# Cell 5: LOAD COCO EVALUATION DATA
# =============================================================================

DATA_DIR = Path("/content/drive/MyDrive/image-captioning/data")
IMG_DIR = DATA_DIR / "train2017"
ANN_FILE = DATA_DIR / "annotations" / "captions_train2017.json"

assert IMG_DIR.exists(), f"Images not found: {IMG_DIR}"
assert ANN_FILE.exists(), f"Annotations not found: {ANN_FILE}"

# Use your Milestone 1 metadata to get known-good image paths
# (avoids scanning 118k files on Drive which causes timeouts)
with open(SAVE_DIR / "milestone1_metadata.json", "r") as f:
    meta = json.load(f)
m1_pairs = meta["pairs"]  # already verified during Milestone 1

# Load COCO annotations for getting ALL captions per image
coco = COCO(str(ANN_FILE))

# Build eval data from your existing 5k pairs (they're guaranteed to exist)
eval_data = []
for p in m1_pairs:
    img_id = p["image_id"]
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)
    all_captions = [a["caption"].strip() for a in anns]
    eval_data.append({
        "image_id": img_id,
        "file_name": Path(p["path"]).name,
        "path": p["path"],
        "captions": all_captions,
    })

# Sample 500 for evaluation (different seed from training split)
random.seed(99)
random.shuffle(eval_data)
eval_data = eval_data[:500]

print(f"[INFO] Evaluation set: {len(eval_data)} images")



In [ ]:
# =============================================================================
# Cell 6: METRIC FUNCTIONS
# =============================================================================

def compute_bleu(references_list, hypotheses):
    refs = [[nltk.word_tokenize(r.lower()) for r in refs] for refs in references_list]
    hyps = [nltk.word_tokenize(h.lower()) for h in hypotheses]
    smooth = SmoothingFunction().method1
    scores = {}
    for n in range(1, 5):
        weights = [1.0 / n] * n + [0.0] * (4 - n)
        scores[f"BLEU-{n}"] = corpus_bleu(refs, hyps, weights=weights,
                                           smoothing_function=smooth)
    return scores


def compute_meteor(references_list, hypotheses):
    scores = []
    for refs, hyp in zip(references_list, hypotheses):
        ref_tok = [nltk.word_tokenize(r.lower()) for r in refs]
        hyp_tok = nltk.word_tokenize(hyp.lower())
        scores.append(meteor_score(ref_tok, hyp_tok))
    return {"METEOR": np.mean(scores)}


def compute_rouge_l(references_list, hypotheses):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = []
    for refs, hyp in zip(references_list, hypotheses):
        best = max(scorer.score(ref, hyp)["rougeL"].fmeasure for ref in refs)
        scores.append(best)
    return {"ROUGE-L": np.mean(scores)}


def compute_cider(references_list, hypotheses):
    try:
        from pycocoevalcap.cider.cider import Cider
        gts = {i: refs for i, refs in enumerate(references_list)}
        res = {i: [hyp] for i, hyp in enumerate(hypotheses)}
        score, _ = Cider().compute_score(gts, res)
        return {"CIDEr": score}
    except ImportError:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        scores = []
        for refs, hyp in zip(references_list, hypotheses):
            corpus = refs + [hyp]
            tfidf = TfidfVectorizer().fit_transform(corpus)
            sims = cosine_similarity(tfidf[-1:], tfidf[:-1])
            scores.append(sims.max())
        return {"CIDEr (approx)": np.mean(scores)}


def evaluate_all_metrics(references_list, hypotheses):
    results = {}
    results.update(compute_bleu(references_list, hypotheses))
    results.update(compute_meteor(references_list, hypotheses))
    results.update(compute_rouge_l(references_list, hypotheses))
    results.update(compute_cider(references_list, hypotheses))
    return results



In [ ]:
# =============================================================================
# Cell 7: GENERATE CAPTIONS & COMPUTE METRICS (3 strategies)
# =============================================================================

print("\n" + "=" * 60)
print("  GENERATING CAPTIONS — 3 Decoding Strategies")
print("=" * 60)

strategies = {
    "Greedy": {"fn": "generate_greedy", "kwargs": {}},
    "Beam Search (k=5)": {"fn": "generate_beam", "kwargs": {"num_beams": 5}},
    "Nucleus (p=0.9)": {"fn": "generate", "kwargs": {"temperature": 0.8, "top_p": 0.9}},
}

all_metrics = {}
all_hypotheses = {}
sample_results = []  # for qualitative eval

for strat_name, strat_config in strategies.items():
    print(f"\n── {strat_name} ──")
    references_list = []
    hypotheses = []
    gen_fn = getattr(model, strat_config["fn"])

    for i, item in enumerate(eval_data):
        img = Image.open(item["path"]).convert("RGB")
        clip_emb = encode_image(img)

        caption = gen_fn(clip_emb, max_new_tokens=40, **strat_config["kwargs"])
        cap_text = caption[0] if isinstance(caption, list) else caption
        cap_text = " ".join(cap_text.split())

        references_list.append(item["captions"])
        hypotheses.append(cap_text)

        # Save detailed results for beam search (for qualitative eval)
        if strat_name == "Beam Search (k=5)":
            sample_results.append({
                "file_name": item["file_name"],
                "path": item["path"],
                "image_id": item["image_id"],
                "generated": cap_text,
                "references": item["captions"],
            })

        if (i + 1) % 50 == 0:
            print(f"   [{i + 1}/{len(eval_data)}] processed")

    metrics = evaluate_all_metrics(references_list, hypotheses)
    all_metrics[strat_name] = metrics
    all_hypotheses[strat_name] = hypotheses
    print(f"   BLEU-4={metrics['BLEU-4']:.4f}  METEOR={metrics['METEOR']:.4f}  "
          f"ROUGE-L={metrics['ROUGE-L']:.4f}")

# ── Print results table ──
print("\n" + "=" * 75)
print("  QUANTITATIVE EVALUATION RESULTS")
print("=" * 75)
header = f"  {'Strategy':<25} {'BLEU-1':>8} {'BLEU-4':>8} {'METEOR':>8} {'ROUGE-L':>8} {'CIDEr':>8}"
print(header)
print("-" * 75)
for name, m in all_metrics.items():
    cider_key = "CIDEr" if "CIDEr" in m else "CIDEr (approx)"
    print(f"  {name:<25} {m['BLEU-1']:>8.4f} {m['BLEU-4']:>8.4f} "
          f"{m['METEOR']:>8.4f} {m['ROUGE-L']:>8.4f} {m[cider_key]:>8.4f}")
print("=" * 75)
print(f"  Evaluated on {len(eval_data)} images")



In [ ]:
print(f"Number of strategies: {len(all_metrics)}")
for name, m in all_metrics.items():
    print(f"\n{name}:")
    for k, v in m.items():
        print(f"  {k}: {v}")

In [ ]:
# Rebuild metrics from the printed results
all_metrics = {
    "Greedy": {"BLEU-1": 0.6693, "BLEU-4": 0.2020, "METEOR": 0.4043, "ROUGE-L": 0.4763, "CIDEr": 0.4716},
    "Beam Search (k=5)": {"BLEU-1": 0.3146, "BLEU-4": 0.0842, "METEOR": 0.3829, "ROUGE-L": 0.3591, "CIDEr": 0.2262},
    "Nucleus (p=0.9)": {"BLEU-1": 0.5885, "BLEU-4": 0.1246, "METEOR": 0.3614, "ROUGE-L": 0.4168, "CIDEr": 0.3431},
}

# Redraw the bar chart
metric_names = ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L"]
strat_names = list(all_metrics.keys())
colors = ["#2ecc71", "#3498db", "#e74c3c"]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(metric_names))
width = 0.25

for i, (sname, color) in enumerate(zip(strat_names, colors)):
    vals = [all_metrics[sname].get(m, 0) for m in metric_names]
    bars = ax.bar(x + i * width, vals, width, label=sname, color=color, edgecolor="white")
    for j, v in enumerate(vals):
        ax.text(x[j] + i * width, v + 0.01, f"{v:.3f}", ha="center", fontsize=7)

ax.set_xlabel("Metric", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Quantitative Metrics by Decoding Strategy", fontsize=14, fontweight="bold")
ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/image-captioning/metrics_comparison.png", dpi=150)
plt.show()
print("[SAVED] metrics_comparison.png")

In [ ]:
# ── Regenerate qualitative samples (10 images) ──
print("Generating qualitative samples...")

sample_results = []
for i, item in enumerate(eval_data[:10]):
    img = Image.open(item["path"]).convert("RGB")
    clip_emb = encode_image(img)
    caption = model.generate(clip_emb, max_new_tokens=40, temperature=0.7, top_p=0.9)
    cap_text = caption[0] if isinstance(caption, list) else caption

    sample_results.append({
        "file_name": item["file_name"],
        "path": item["path"],
        "image_id": item["image_id"],
        "generated": cap_text.strip(),
        "references": item["captions"],
    })
    print(f"  [{i+1}/10] GEN: {cap_text.strip()[:70]}")
    print(f"           REF: {item['captions'][0][:70]}\n")

# ── Qualitative grid ──
n = len(sample_results)
cols = min(5, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 5.5 * rows))
axes = np.array(axes).flatten()

for idx in range(len(axes)):
    ax = axes[idx]
    if idx < n:
        item = sample_results[idx]
        try:
            img = Image.open(item["path"]).convert("RGB")
            ax.imshow(img)
        except:
            ax.text(0.5, 0.5, "Image not found", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"GEN: {item['generated'][:60]}", fontsize=8, color="green", pad=4, wrap=True)
        ax.set_xlabel(f"REF: {item['references'][0][:60]}", fontsize=7, color="gray", wrap=True)
    ax.set_xticks([])
    ax.set_yticks([])
    if idx >= n:
        ax.axis("off")

plt.suptitle("Qualitative: Generated (green) vs Reference (gray)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/image-captioning/qualitative_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print("[SAVED] qualitative_evaluation.png")

# ── Per-sample METEOR distribution ──
meteor_scores = []
for item in sample_results:
    ref_tok = [nltk.word_tokenize(r.lower()) for r in item["references"]]
    hyp_tok = nltk.word_tokenize(item["generated"].lower())
    meteor_scores.append(meteor_score(ref_tok, hyp_tok))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(meteor_scores, bins=10, edgecolor="black", alpha=0.7, color="steelblue")
ax.axvline(np.mean(meteor_scores), color="red", linestyle="--", label=f"Mean = {np.mean(meteor_scores):.4f}")
ax.set_xlabel("METEOR Score")
ax.set_ylabel("Count")
ax.set_title("Per-Sample METEOR Score Distribution")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/image-captioning/meteor_distribution.png", dpi=150)
plt.show()
print("[SAVED] meteor_distribution.png")

print("\n✅ Task 1 fully complete!")

In [ ]:
"""
=============================================================================
MILESTONE 3 — TASK 2
Analyze parameter sensitivity (temperature, beam width, embedding injection)
Compare fine-tuning strategies (full fine-tuning vs. LoRA vs. prefix tuning)
=============================================================================
Assumes Task 1 cells already ran (model, tokenizer, CLIP, eval_data loaded)
=============================================================================
"""

# ── Cell 1: Install LoRA library ──
# !pip install peft

# ── Cell 2: Imports (skip if already loaded from Task 1) ──
import os, json, random, time, warnings, re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer, CLIPModel, CLIPProcessor
from pycocotools.coco import COCO
from peft import LoraConfig, get_peft_model, TaskType

import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device: {DEVICE}")


# ── Cell 3: Load tokenizer, CLIP, data (skip if already in memory from Task 1) ──
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SAVE_DIR = Path("/content/drive/MyDrive/image-captioning/processed")
tok_path = str(SAVE_DIR / "tokenizer")
tokenizer = GPT2Tokenizer.from_pretrained(tok_path, local_files_only=True)
print(f"[INFO] Tokenizer: vocab size {len(tokenizer)}")

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model.eval()
print(f"[INFO] CLIP loaded: {CLIP_MODEL_NAME}")

@torch.no_grad()
def encode_image(image):
    inputs = clip_processor(images=image, return_tensors="pt").to(DEVICE)
    outputs = clip_model.get_image_features(**inputs)
    if isinstance(outputs, torch.Tensor):
        feats = outputs
    else:
        feats = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs[0]
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats

# Load eval data from Milestone 1 metadata
with open(SAVE_DIR / "milestone1_metadata.json", "r") as f:
    meta = json.load(f)
m1_pairs = meta["pairs"]

DATA_DIR = Path("/content/drive/MyDrive/image-captioning/data")
ANN_FILE = DATA_DIR / "annotations" / "captions_train2017.json"
coco = COCO(str(ANN_FILE))

eval_data = []
for p in m1_pairs:
    img_id = p["image_id"]
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)
    all_captions = [a["caption"].strip() for a in anns]
    eval_data.append({
        "image_id": img_id,
        "file_name": Path(p["path"]).name,
        "path": p["path"],
        "captions": all_captions,
    })
random.seed(99)
random.shuffle(eval_data)
eval_data = eval_data[:200]  # 200 images for faster sweeps
print(f"[INFO] Eval set: {len(eval_data)} images")

# Load training data
data = torch.load(SAVE_DIR / "milestone1_tensors.pt", weights_only=True)
image_embeddings = data["image_embeddings"]
caption_input_ids = data["caption_input_ids"]
caption_attention_masks = data["caption_attention_masks"]

indices = list(range(len(image_embeddings)))
random.seed(42)
random.shuffle(indices)
train_idx = indices[:4500]
print(f"[INFO] Training samples: {len(train_idx)}")


# ── Cell 4: Model + metrics (same as Task 1) ──

class PrefixProjection(nn.Module):
    def __init__(self, clip_dim=512, gpt_dim=768, num_prefix_tokens=8):
        super().__init__()
        self.num_prefix_tokens = num_prefix_tokens
        self.gpt_dim = gpt_dim
        hidden_dim = gpt_dim * 2
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, gpt_dim * num_prefix_tokens),
            nn.LayerNorm(gpt_dim * num_prefix_tokens),
        )

    def forward(self, clip_emb):
        return self.proj(clip_emb).view(-1, self.num_prefix_tokens, self.gpt_dim)


class CLIPGpt2CaptionModel(nn.Module):
    def __init__(self, num_prefix_tokens=8, unfreeze_last_n=2):
        super().__init__()
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt2.resize_token_embeddings(len(tokenizer))
        gpt_dim = self.gpt2.config.n_embd
        self.prefix_proj = PrefixProjection(512, gpt_dim, num_prefix_tokens)
        self.num_prefix = num_prefix_tokens

        # Freeze all first
        for param in self.gpt2.parameters():
            param.requires_grad = False
        # Unfreeze last N layers + heads
        for i in range(self.gpt2.config.n_layer - unfreeze_last_n, self.gpt2.config.n_layer):
            for param in self.gpt2.transformer.h[i].parameters():
                param.requires_grad = True
        for param in self.gpt2.lm_head.parameters():
            param.requires_grad = True
        for param in self.gpt2.transformer.ln_f.parameters():
            param.requires_grad = True
        for param in self.gpt2.transformer.wte.parameters():
            param.requires_grad = True

    def forward(self, clip_embedding, input_ids, attention_mask):
        B, T = input_ids.shape
        prefix_embeds = self.prefix_proj(clip_embedding)
        text_embeds = self.gpt2.transformer.wte(input_ids)
        inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)

        prefix_mask = torch.ones(B, self.num_prefix, device=input_ids.device)
        full_mask = torch.cat([prefix_mask, attention_mask.float()], dim=1)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        prefix_labels = torch.full((B, self.num_prefix), -100, dtype=torch.long, device=input_ids.device)
        full_labels = torch.cat([prefix_labels, labels], dim=1)

        outputs = self.gpt2(inputs_embeds=inputs_embeds, attention_mask=full_mask, labels=full_labels)
        return outputs.loss, outputs.logits

    @torch.no_grad()
    def generate(self, clip_embedding, max_new_tokens=50, temperature=0.8, top_p=0.9):
        self.eval()
        if clip_embedding.dim() == 1:
            clip_embedding = clip_embedding.unsqueeze(0)
        B = clip_embedding.shape[0]
        device = clip_embedding.device
        prefix_embeds = self.prefix_proj(clip_embedding)
        cur_ids = torch.full((B, 1), tokenizer.bos_token_id, device=device)

        for _ in range(max_new_tokens):
            text_embeds = self.gpt2.transformer.wte(cur_ids)
            embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
            outputs = self.gpt2(inputs_embeds=embeds)
            next_logits = outputs.logits[:, -1, :] / temperature
            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
            cumulative = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cumulative - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = -float("inf")
            probs = F.softmax(sorted_logits, dim=-1)
            sampled = torch.multinomial(probs, 1)
            next_token = sorted_idx.gather(-1, sampled)
            cur_ids = torch.cat([cur_ids, next_token], dim=1)
            if (next_token == tokenizer.eos_token_id).all():
                break
        return [c.strip() for c in tokenizer.batch_decode(cur_ids, skip_special_tokens=True)]

    @torch.no_grad()
    def generate_greedy(self, clip_embedding, max_new_tokens=50):
        self.eval()
        if clip_embedding.dim() == 1:
            clip_embedding = clip_embedding.unsqueeze(0)
        B = clip_embedding.shape[0]
        device = clip_embedding.device
        prefix_embeds = self.prefix_proj(clip_embedding)
        cur_ids = torch.full((B, 1), tokenizer.bos_token_id, device=device)
        for _ in range(max_new_tokens):
            text_embeds = self.gpt2.transformer.wte(cur_ids)
            embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
            outputs = self.gpt2(inputs_embeds=embeds)
            next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            cur_ids = torch.cat([cur_ids, next_token], dim=1)
            if (next_token == tokenizer.eos_token_id).all():
                break
        return [c.strip() for c in tokenizer.batch_decode(cur_ids, skip_special_tokens=True)]

    @torch.no_grad()
    def generate_beam(self, clip_embedding, max_new_tokens=50, num_beams=5):
        self.eval()
        if clip_embedding.dim() == 1:
            clip_embedding = clip_embedding.unsqueeze(0)
        device = clip_embedding.device
        prefix_embeds = self.prefix_proj(clip_embedding)
        beams = [(torch.tensor([[tokenizer.bos_token_id]], device=device), 0.0)]
        completed = []
        for _ in range(max_new_tokens):
            candidates = []
            for seq, score in beams:
                text_embeds = self.gpt2.transformer.wte(seq)
                embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
                log_probs = F.log_softmax(self.gpt2(inputs_embeds=embeds).logits[:, -1, :], dim=-1)
                top_lp, top_ids = log_probs.topk(num_beams, dim=-1)
                for i in range(num_beams):
                    tid = top_ids[0, i].item()
                    ns = score + top_lp[0, i].item()
                    nseq = torch.cat([seq, torch.tensor([[tid]], device=device)], dim=1)
                    (completed if tid == tokenizer.eos_token_id else candidates).append((nseq, ns))
            if not candidates:
                break
            candidates.sort(key=lambda x: x[1], reverse=True)
            beams = candidates[:num_beams]
        all_opts = completed + beams
        all_opts.sort(key=lambda x: x[1] / x[0].size(1), reverse=True)
        return tokenizer.batch_decode(all_opts[0][0], skip_special_tokens=True)


def compute_metrics(references_list, hypotheses):
    refs = [[nltk.word_tokenize(r.lower()) for r in refs] for refs in references_list]
    hyps = [nltk.word_tokenize(h.lower()) for h in hypotheses]
    smooth = SmoothingFunction().method1
    results = {}
    for n in range(1, 5):
        w = [1.0/n]*n + [0.0]*(4-n)
        results[f"BLEU-{n}"] = corpus_bleu(refs, hyps, weights=w, smoothing_function=smooth)

    m_scores = []
    for rfs, hyp in zip(references_list, hypotheses):
        rt = [nltk.word_tokenize(r.lower()) for r in rfs]
        ht = nltk.word_tokenize(hyp.lower())
        m_scores.append(meteor_score(rt, ht))
    results["METEOR"] = np.mean(m_scores)

    sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    r_scores = []
    for rfs, hyp in zip(references_list, hypotheses):
        best = max(sc.score(r, hyp)["rougeL"].fmeasure for r in rfs)
        r_scores.append(best)
    results["ROUGE-L"] = np.mean(r_scores)
    return results


def evaluate_with_params(model, eval_data, gen_fn_name, gen_kwargs, n=200):
    """Generate captions and compute metrics."""
    model.eval()
    refs_list, hyps = [], []
    gen_fn = getattr(model, gen_fn_name)
    subset = eval_data[:n]

    for item in subset:
        try:
            img = Image.open(item["path"]).convert("RGB")
            clip_emb = encode_image(img)
            cap = gen_fn(clip_emb, max_new_tokens=40, **gen_kwargs)
            cap_text = cap[0] if isinstance(cap, list) else cap
            refs_list.append(item["captions"])
            hyps.append(cap_text.strip())
        except Exception as e:
            continue

    return compute_metrics(refs_list, hyps)


# =============================================================================
# Cell 5: LOAD TRAINED MODEL
# =============================================================================

CHECKPOINT_PATH = "/content/drive/MyDrive/image-captioning/models/best_model.pt"
model = CLIPGpt2CaptionModel(num_prefix_tokens=8, unfreeze_last_n=2).to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"[INFO] Loaded trained model (epoch {ckpt['epoch']+1}, val_loss {ckpt['val_loss']:.4f})")


# =============================================================================
# Cell 6: TEMPERATURE SWEEP
# =============================================================================

print("\n" + "=" * 60)
print("  SWEEP 1: Temperature (nucleus sampling, top_p=0.9)")
print("=" * 60)

temperatures = [0.3, 0.5, 0.7, 0.9, 1.0, 1.2, 1.5]
temp_results = {}

for temp in temperatures:
    print(f"\n  Temperature = {temp}")
    metrics = evaluate_with_params(
        model, eval_data, "generate",
        {"temperature": temp, "top_p": 0.9}, n=200
    )
    temp_results[temp] = metrics
    print(f"    BLEU-4={metrics['BLEU-4']:.4f}  METEOR={metrics['METEOR']:.4f}  ROUGE-L={metrics['ROUGE-L']:.4f}")


# =============================================================================
# Cell 7: BEAM WIDTH SWEEP
# =============================================================================

print("\n" + "=" * 60)
print("  SWEEP 2: Beam Width")
print("=" * 60)

beam_widths = [1, 3, 5, 8, 10]
beam_results = {}

for bw in beam_widths:
    print(f"\n  Beam width = {bw}")
    metrics = evaluate_with_params(
        model, eval_data, "generate_beam",
        {"num_beams": bw}, n=200
    )
    beam_results[bw] = metrics
    print(f"    BLEU-4={metrics['BLEU-4']:.4f}  METEOR={metrics['METEOR']:.4f}  ROUGE-L={metrics['ROUGE-L']:.4f}")


# =============================================================================
# Cell 8: PLOT PARAMETER SWEEPS
# =============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Temperature plot
temps = sorted(temp_results.keys())
for metric_name, color, marker in [("BLEU-4", "#e74c3c", "o"), ("METEOR", "#3498db", "s"), ("ROUGE-L", "#2ecc71", "^")]:
    vals = [temp_results[t][metric_name] for t in temps]
    ax1.plot(temps, vals, f"{marker}-", label=metric_name, color=color, markersize=6)
ax1.set_xlabel("Temperature", fontsize=12)
ax1.set_ylabel("Score", fontsize=12)
ax1.set_title("Effect of Temperature on Caption Quality", fontsize=13, fontweight="bold")
ax1.legend()
ax1.grid(alpha=0.3)

# Beam width plot
beams = sorted(beam_results.keys())
for metric_name, color, marker in [("BLEU-4", "#e74c3c", "o"), ("METEOR", "#3498db", "s"), ("ROUGE-L", "#2ecc71", "^")]:
    vals = [beam_results[b][metric_name] for b in beams]
    ax2.plot(beams, vals, f"{marker}-", label=metric_name, color=color, markersize=6)
ax2.set_xlabel("Beam Width", fontsize=12)
ax2.set_ylabel("Score", fontsize=12)
ax2.set_title("Effect of Beam Width on Caption Quality", fontsize=13, fontweight="bold")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/image-captioning/parameter_sweeps.png", dpi=150)
plt.show()
print("[SAVED] parameter_sweeps.png")


# =============================================================================
# Cell 9: FINE-TUNING STRATEGY COMPARISON
# =============================================================================

def quick_train(model, train_idx, epochs=3, lr=5e-5, label=""):
    """Train for a few epochs, return loss history."""
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.01
    )
    scaler = torch.amp.GradScaler("cuda") if DEVICE.type == "cuda" else None
    losses = []
    BATCH_SIZE = 32

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0
        random.shuffle(train_idx)

        for i in range(0, min(len(train_idx), 1500), BATCH_SIZE):
            batch_idx = train_idx[i:i+BATCH_SIZE]
            clip_emb = image_embeddings[batch_idx].to(DEVICE)
            ids = caption_input_ids[batch_idx].to(DEVICE)
            mask = caption_attention_masks[batch_idx].to(DEVICE)

            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    loss, _ = model(clip_emb, ids, mask)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss, _ = model(clip_emb, ids, mask)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg = epoch_loss / max(n_batches, 1)
        losses.append(avg)
        print(f"    [{label}] Epoch {epoch+1}/{epochs} — Loss: {avg:.4f}")

    return losses


print("\n" + "=" * 60)
print("  FINE-TUNING STRATEGY COMPARISON")
print("=" * 60)

ft_results = {}

# ── Strategy 1: Prefix Tuning Only (freeze all GPT-2) ──
print("\n[1/3] Prefix Tuning Only...")
model_prefix = CLIPGpt2CaptionModel(num_prefix_tokens=8, unfreeze_last_n=0).to(DEVICE)
# Make sure ONLY prefix_proj is trainable
for p in model_prefix.gpt2.parameters():
    p.requires_grad = False
for p in model_prefix.prefix_proj.parameters():
    p.requires_grad = True

trainable_p = sum(p.numel() for p in model_prefix.parameters() if p.requires_grad)
print(f"    Trainable params: {trainable_p/1e6:.2f}M")

loss_prefix = quick_train(model_prefix, train_idx, epochs=3, label="Prefix Only")
metrics_prefix = evaluate_with_params(model_prefix, eval_data, "generate", {"temperature": 0.7, "top_p": 0.9}, n=200)
ft_results["Prefix Tuning"] = {"metrics": metrics_prefix, "losses": loss_prefix, "params": trainable_p}
print(f"    BLEU-4={metrics_prefix['BLEU-4']:.4f}  METEOR={metrics_prefix['METEOR']:.4f}")

del model_prefix
torch.cuda.empty_cache()

# ── Strategy 2: Full Fine-Tuning (all GPT-2 trainable) ──
print("\n[2/3] Full Fine-Tuning...")
model_full = CLIPGpt2CaptionModel(num_prefix_tokens=8, unfreeze_last_n=12).to(DEVICE)
# Unfreeze everything
for p in model_full.parameters():
    p.requires_grad = True

trainable_f = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
print(f"    Trainable params: {trainable_f/1e6:.2f}M")

loss_full = quick_train(model_full, train_idx, epochs=3, lr=2e-5, label="Full FT")
metrics_full = evaluate_with_params(model_full, eval_data, "generate", {"temperature": 0.7, "top_p": 0.9}, n=200)
ft_results["Full Fine-Tuning"] = {"metrics": metrics_full, "losses": loss_full, "params": trainable_f}
print(f"    BLEU-4={metrics_full['BLEU-4']:.4f}  METEOR={metrics_full['METEOR']:.4f}")

del model_full
torch.cuda.empty_cache()

# ── Strategy 3: LoRA (low-rank adaptation on GPT-2 attention) ──
print("\n[3/3] LoRA Fine-Tuning...")
model_lora = CLIPGpt2CaptionModel(num_prefix_tokens=8, unfreeze_last_n=0).to(DEVICE)
# Freeze everything first
for p in model_lora.gpt2.parameters():
    p.requires_grad = False
for p in model_lora.prefix_proj.parameters():
    p.requires_grad = True

# Apply LoRA to GPT-2 attention layers
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn", "c_proj"],
)
model_lora.gpt2 = get_peft_model(model_lora.gpt2, lora_config)
model_lora.gpt2.print_trainable_parameters()

trainable_l = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"    Total trainable (LoRA + prefix): {trainable_l/1e6:.2f}M")

loss_lora = quick_train(model_lora, train_idx, epochs=3, label="LoRA")
metrics_lora = evaluate_with_params(model_lora, eval_data, "generate", {"temperature": 0.7, "top_p": 0.9}, n=200)
ft_results["LoRA"] = {"metrics": metrics_lora, "losses": loss_lora, "params": trainable_l}
print(f"    BLEU-4={metrics_lora['BLEU-4']:.4f}  METEOR={metrics_lora['METEOR']:.4f}")

del model_lora
torch.cuda.empty_cache()


# =============================================================================
# Cell 10: RESULTS TABLE
# =============================================================================

print("\n" + "=" * 80)
print("  FINE-TUNING STRATEGY COMPARISON — RESULTS")
print("=" * 80)
header = f"  {'Strategy':<22} {'Params':>10} {'BLEU-1':>8} {'BLEU-4':>8} {'METEOR':>8} {'ROUGE-L':>8}"
print(header)
print("-" * 80)
for name, data in ft_results.items():
    m = data["metrics"]
    p = data["params"]
    print(f"  {name:<22} {p/1e6:>8.2f}M {m['BLEU-1']:>8.4f} {m['BLEU-4']:>8.4f} "
          f"{m['METEOR']:>8.4f} {m['ROUGE-L']:>8.4f}")
print("=" * 80)


# =============================================================================
# Cell 11: PLOT FINE-TUNING COMPARISON
# =============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Metric comparison bars ──
strategies = list(ft_results.keys())
metric_names = ["BLEU-4", "METEOR", "ROUGE-L"]
colors = ["#2ecc71", "#3498db", "#e74c3c"]
x = np.arange(len(metric_names))
width = 0.25

for i, (strat, color) in enumerate(zip(strategies, colors)):
    vals = [ft_results[strat]["metrics"][m] for m in metric_names]
    bars = ax1.bar(x + i * width, vals, width, label=strat, color=color, edgecolor="white")
    for j, v in enumerate(vals):
        ax1.text(x[j] + i * width, v + 0.005, f"{v:.3f}", ha="center", fontsize=7)

ax1.set_xticks(x + width)
ax1.set_xticklabels(metric_names)
ax1.set_ylabel("Score")
ax1.set_title("Fine-Tuning Strategy: Metric Comparison", fontweight="bold")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# ── Training loss curves ──
for strat, color in zip(strategies, colors):
    losses = ft_results[strat]["losses"]
    ax2.plot(range(1, len(losses)+1), losses, "o-", label=strat, color=color)

ax2.set_xlabel("Epoch")
ax2.set_ylabel("Training Loss")
ax2.set_title("Training Loss Curves", fontweight="bold")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/image-captioning/finetuning_comparison.png", dpi=150)
plt.show()
print("[SAVED] finetuning_comparison.png")


# =============================================================================
# Cell 12: SAVE ALL TASK 2 RESULTS
# =============================================================================

task2_output = {
    "experiment": "milestone3_task2_parameter_sensitivity",
    "timestamp": datetime.now().isoformat(),
    "temperature_sweep": {str(k): v for k, v in temp_results.items()},
    "beam_width_sweep": {str(k): v for k, v in beam_results.items()},
    "finetuning_comparison": {
        name: {"metrics": data["metrics"], "losses": data["losses"],
               "trainable_params": data["params"]}
        for name, data in ft_results.items()
    },
}

save_path = "/content/drive/MyDrive/image-captioning/task2_parameter_results.json"
with open(save_path, "w") as f:
    json.dump(task2_output, f, indent=2, default=str)

print(f"\n[SAVED] {save_path}")
print("\n" + "=" * 55)
print("  Task 2 Complete!")
print("=" * 55)
print("  Files generated:")
print("    - parameter_sweeps.png          (temperature + beam width)")
print("    - finetuning_comparison.png     (bars + loss curves)")
print("    - task2_parameter_results.json   (all numeric results)")

In [ ]:
"""
=============================================================================
MILESTONE 3 — TASK 3
Visualize & compare results across decoding strategies
=============================================================================
Assumes Tasks 1 & 2 already ran. If variables are lost from memory,
this code reloads everything from saved JSON files.
=============================================================================
"""

# ── Cell 1: Imports & Setup ──
import json, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer, CLIPModel, CLIPProcessor

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE_DIR = "/content/drive/MyDrive/image-captioning"
SAVE_DIR = Path(f"{BASE_DIR}/processed")
print(f"[INFO] Device: {DEVICE}")


# ── Cell 2: Reload Task 1 & Task 2 results from JSON ──

# Task 1 metrics
try:
    with open(f"{BASE_DIR}/task1_metrics_results.json") as f:
        task1_data = json.load(f)
    all_metrics = task1_data.get("metrics_by_strategy", {})
    print(f"[INFO] Task 1 loaded: {len(all_metrics)} strategies")
except FileNotFoundError:
    print("[WARN] task1_metrics_results.json not found — using hardcoded values")
    all_metrics = {
        "Greedy": {"BLEU-1": 0.6693, "BLEU-4": 0.2020, "METEOR": 0.4043, "ROUGE-L": 0.4763, "CIDEr": 0.4716},
        "Beam Search (k=5)": {"BLEU-1": 0.3146, "BLEU-4": 0.0842, "METEOR": 0.3829, "ROUGE-L": 0.3591, "CIDEr": 0.2262},
        "Nucleus (p=0.9)": {"BLEU-1": 0.5885, "BLEU-4": 0.1246, "METEOR": 0.3614, "ROUGE-L": 0.4168, "CIDEr": 0.3431},
    }

# Task 2 results
try:
    with open(f"{BASE_DIR}/task2_parameter_results.json") as f:
        task2_data = json.load(f)
    temp_results = {float(k): v for k, v in task2_data.get("temperature_sweep", {}).items()}
    beam_results = {int(k): v for k, v in task2_data.get("beam_width_sweep", {}).items()}
    ft_results = task2_data.get("finetuning_comparison", {})
    print(f"[INFO] Task 2 loaded: {len(temp_results)} temps, {len(beam_results)} beams, {len(ft_results)} FT strategies")
except FileNotFoundError:
    print("[WARN] task2_parameter_results.json not found — some plots will be skipped")
    temp_results, beam_results, ft_results = {}, {}, {}


# =============================================================================
# Cell 3: COMPREHENSIVE DECODING STRATEGY COMPARISON TABLE
# =============================================================================

print("\n" + "=" * 85)
print("  COMPREHENSIVE RESULTS — ALL DECODING STRATEGIES")
print("=" * 85)
header = f"  {'Strategy':<25} {'BLEU-1':>8} {'BLEU-4':>8} {'METEOR':>8} {'ROUGE-L':>8} {'CIDEr':>8}"
print(header)
print("-" * 85)
for name, m in all_metrics.items():
    cider_key = "CIDEr" if "CIDEr" in m else "CIDEr (approx)"
    cider_val = m.get(cider_key, m.get("CIDEr", 0))
    print(f"  {name:<25} {m['BLEU-1']:>8.4f} {m['BLEU-4']:>8.4f} "
          f"{m['METEOR']:>8.4f} {m['ROUGE-L']:>8.4f} {cider_val:>8.4f}")
print("=" * 85)

# Find best strategy per metric
print("\n  Best strategy per metric:")
for metric in ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L"]:
    best_name = max(all_metrics, key=lambda s: all_metrics[s].get(metric, 0))
    best_val = all_metrics[best_name][metric]
    print(f"    {metric:<10} → {best_name} ({best_val:.4f})")


# =============================================================================
# Cell 4: BAR CHART — Decoding Strategies Side by Side
# =============================================================================

metric_names = ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L"]
strat_names = list(all_metrics.keys())
colors = ["#2ecc71", "#3498db", "#e74c3c", "#9b59b6", "#f39c12"]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metric_names))
width = 0.8 / len(strat_names)

for i, sname in enumerate(strat_names):
    vals = [all_metrics[sname].get(m, 0) for m in metric_names]
    bars = ax.bar(x + i * width, vals, width, label=sname,
                  color=colors[i % len(colors)], edgecolor="white", linewidth=0.5)
    for j, v in enumerate(vals):
        ax.text(x[j] + i * width, v + 0.008, f"{v:.3f}", ha="center", fontsize=7, fontweight="bold")

ax.set_xlabel("Metric", fontsize=13)
ax.set_ylabel("Score", fontsize=13)
ax.set_title("Decoding Strategy Comparison — All Metrics", fontsize=15, fontweight="bold")
ax.set_xticks(x + width * (len(strat_names) - 1) / 2)
ax.set_xticklabels(metric_names, fontsize=11)
ax.legend(loc="upper right", fontsize=10)
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, max(max(all_metrics[s].get(m, 0) for m in metric_names) for s in strat_names) * 1.18)

plt.tight_layout()
plt.savefig(f"{BASE_DIR}/task3_decoding_comparison.png", dpi=150)
plt.show()
print("[SAVED] task3_decoding_comparison.png")


# =============================================================================
# Cell 5: RADAR CHART — Strategy Strengths & Weaknesses
# =============================================================================

def radar_chart(all_metrics, title="Decoding Strategy Radar"):
    metrics = ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L"]
    N = len(metrics)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    colors = ["#2ecc71", "#3498db", "#e74c3c"]

    for i, (sname, m) in enumerate(all_metrics.items()):
        values = [m.get(metric, 0) for metric in metrics]
        values += values[:1]
        ax.plot(angles, values, "o-", linewidth=2, label=sname,
                color=colors[i % len(colors)], markersize=6)
        ax.fill(angles, values, alpha=0.1, color=colors[i % len(colors)])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=11)
    ax.set_title(title, fontsize=14, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/task3_radar_chart.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("[SAVED] task3_radar_chart.png")

radar_chart(all_metrics)


# =============================================================================
# Cell 6: SIDE-BY-SIDE CAPTION COMPARISON (same image, all strategies)
# =============================================================================

# Load model + CLIP for generating fresh captions
tok_path = str(SAVE_DIR / "tokenizer")
tokenizer = GPT2Tokenizer.from_pretrained(tok_path, local_files_only=True)

clip_model_hf = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_processor_hf = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model_hf.eval()

@torch.no_grad()
def encode_image(image):
    inputs = clip_processor_hf(images=image, return_tensors="pt").to(DEVICE)
    outputs = clip_model_hf.get_image_features(**inputs)
    if isinstance(outputs, torch.Tensor):
        feats = outputs
    else:
        feats = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs[0]
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats


class PrefixProjection(nn.Module):
    def __init__(self, clip_dim=512, gpt_dim=768, num_prefix_tokens=8):
        super().__init__()
        self.num_prefix_tokens = num_prefix_tokens
        self.gpt_dim = gpt_dim
        hidden_dim = gpt_dim * 2
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, gpt_dim * num_prefix_tokens),
            nn.LayerNorm(gpt_dim * num_prefix_tokens),
        )
    def forward(self, clip_emb):
        return self.proj(clip_emb).view(-1, self.num_prefix_tokens, self.gpt_dim)


class CLIPGpt2CaptionModel(nn.Module):
    def __init__(self, num_prefix_tokens=8, unfreeze_last_n=2):
        super().__init__()
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt2.resize_token_embeddings(len(tokenizer))
        gpt_dim = self.gpt2.config.n_embd
        self.prefix_proj = PrefixProjection(512, gpt_dim, num_prefix_tokens)
        self.num_prefix = num_prefix_tokens
        for param in self.gpt2.parameters():
            param.requires_grad = False
        for i in range(self.gpt2.config.n_layer - unfreeze_last_n, self.gpt2.config.n_layer):
            for param in self.gpt2.transformer.h[i].parameters():
                param.requires_grad = True
        for param in self.gpt2.lm_head.parameters():
            param.requires_grad = True
        for param in self.gpt2.transformer.ln_f.parameters():
            param.requires_grad = True
        for param in self.gpt2.transformer.wte.parameters():
            param.requires_grad = True

    def _gen_loop(self, clip_emb, max_new_tokens, get_next_token_fn):
        self.eval()
        if clip_emb.dim() == 1:
            clip_emb = clip_emb.unsqueeze(0)
        prefix = self.prefix_proj(clip_emb)
        cur = torch.full((clip_emb.shape[0], 1), tokenizer.bos_token_id, device=clip_emb.device)
        for _ in range(max_new_tokens):
            embeds = torch.cat([prefix, self.gpt2.transformer.wte(cur)], dim=1)
            logits = self.gpt2(inputs_embeds=embeds).logits[:, -1, :]
            next_tok = get_next_token_fn(logits)
            cur = torch.cat([cur, next_tok], dim=1)
            if (next_tok == tokenizer.eos_token_id).all():
                break
        return [c.strip() for c in tokenizer.batch_decode(cur, skip_special_tokens=True)]

    @torch.no_grad()
    def generate_greedy(self, clip_emb, max_new_tokens=40):
        return self._gen_loop(clip_emb, max_new_tokens,
            lambda logits: logits.argmax(dim=-1, keepdim=True))

    @torch.no_grad()
    def generate_nucleus(self, clip_emb, max_new_tokens=40, temperature=0.8, top_p=0.9):
        def sample(logits):
            logits = logits / temperature
            sl, si = torch.sort(logits, descending=True)
            cum = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
            mask = cum - F.softmax(sl, dim=-1) >= top_p
            sl[mask] = -float("inf")
            return si.gather(-1, torch.multinomial(F.softmax(sl, dim=-1), 1))
        return self._gen_loop(clip_emb, max_new_tokens, sample)

    @torch.no_grad()
    def generate_beam(self, clip_emb, max_new_tokens=40, num_beams=5):
        self.eval()
        if clip_emb.dim() == 1:
            clip_emb = clip_emb.unsqueeze(0)
        device = clip_emb.device
        prefix = self.prefix_proj(clip_emb)
        beams = [(torch.tensor([[tokenizer.bos_token_id]], device=device), 0.0)]
        completed = []
        for _ in range(max_new_tokens):
            cands = []
            for seq, score in beams:
                embeds = torch.cat([prefix, self.gpt2.transformer.wte(seq)], dim=1)
                lp = F.log_softmax(self.gpt2(inputs_embeds=embeds).logits[:, -1, :], dim=-1)
                top_lp, top_ids = lp.topk(num_beams, dim=-1)
                for i in range(num_beams):
                    tid = top_ids[0, i].item()
                    ns = score + top_lp[0, i].item()
                    nseq = torch.cat([seq, torch.tensor([[tid]], device=device)], dim=1)
                    (completed if tid == tokenizer.eos_token_id else cands).append((nseq, ns))
            if not cands:
                break
            cands.sort(key=lambda x: x[1], reverse=True)
            beams = cands[:num_beams]
        opts = completed + beams
        opts.sort(key=lambda x: x[1] / x[0].size(1), reverse=True)
        return tokenizer.batch_decode(opts[0][0], skip_special_tokens=True)


# Load trained model
model = CLIPGpt2CaptionModel(num_prefix_tokens=8, unfreeze_last_n=2).to(DEVICE)
ckpt = torch.load(f"{BASE_DIR}/models/best_model.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"[INFO] Model loaded (epoch {ckpt['epoch']+1})")

# Load eval data
with open(SAVE_DIR / "milestone1_metadata.json") as f:
    m1_meta = json.load(f)
from pycocotools.coco import COCO
ANN_FILE = f"{BASE_DIR}/data/annotations/captions_train2017.json"
coco = COCO(ANN_FILE)

sample_pairs = random.sample(m1_meta["pairs"], 5)
print(f"[INFO] Generating captions for 5 images with 3 strategies each...\n")

comparison_data = []
for idx, p in enumerate(sample_pairs):
    img = Image.open(p["path"]).convert("RGB")
    clip_emb = encode_image(img)

    ann_ids = coco.getAnnIds(imgIds=p["image_id"])
    gt_caps = [a["caption"].strip() for a in coco.loadAnns(ann_ids)]

    cap_greedy = model.generate_greedy(clip_emb)[0]
    cap_beam = model.generate_beam(clip_emb, num_beams=5)[0]
    cap_nucleus = model.generate_nucleus(clip_emb, temperature=0.8, top_p=0.9)[0]

    comparison_data.append({
        "path": p["path"],
        "file_name": Path(p["path"]).name,
        "ground_truth": gt_caps[0],
        "greedy": cap_greedy,
        "beam_search": cap_beam,
        "nucleus": cap_nucleus,
    })

    print(f"Image {idx+1}: {Path(p['path']).name}")
    print(f"  GT:      {gt_caps[0][:70]}")
    print(f"  Greedy:  {cap_greedy[:70]}")
    print(f"  Beam:    {cap_beam[:70]}")
    print(f"  Nucleus: {cap_nucleus[:70]}\n")

# ── Visual comparison grid ──
fig, axes = plt.subplots(len(comparison_data), 4, figsize=(20, 5 * len(comparison_data)),
                          gridspec_kw={"width_ratios": [1.5, 1, 1, 1]})

for row, item in enumerate(comparison_data):
    # Image
    img = Image.open(item["path"]).convert("RGB")
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"GT: {item['ground_truth'][:50]}", fontsize=8, color="green", wrap=True)
    axes[row, 0].axis("off")

    # Captions as text panels
    for col, (strat, key, color) in enumerate([
        ("Greedy", "greedy", "#2ecc71"),
        ("Beam Search", "beam_search", "#3498db"),
        ("Nucleus", "nucleus", "#e74c3c"),
    ]):
        axes[row, col+1].text(0.5, 0.5, item[key][:100],
            ha="center", va="center", fontsize=9, wrap=True,
            transform=axes[row, col+1].transAxes,
            bbox=dict(boxstyle="round,pad=0.5", facecolor=color, alpha=0.15))
        if row == 0:
            axes[row, col+1].set_title(strat, fontsize=12, fontweight="bold", color=color)
        axes[row, col+1].axis("off")

plt.suptitle("Same Image — Different Decoding Strategies",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{BASE_DIR}/task3_caption_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("[SAVED] task3_caption_comparison.png")


# =============================================================================
# Cell 7: COMBINED PARAMETER SENSITIVITY PLOTS (from Task 2)
# =============================================================================

if temp_results and beam_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Temperature
    temps = sorted(temp_results.keys())
    for m, c, mk in [("BLEU-4", "#e74c3c", "o"), ("METEOR", "#3498db", "s"), ("ROUGE-L", "#2ecc71", "^")]:
        vals = [temp_results[t][m] for t in temps]
        axes[0].plot(temps, vals, f"{mk}-", label=m, color=c, markersize=7, linewidth=2)
    axes[0].set_xlabel("Temperature", fontsize=12)
    axes[0].set_ylabel("Score", fontsize=12)
    axes[0].set_title("Temperature Sensitivity", fontsize=13, fontweight="bold")
    axes[0].legend(fontsize=10)
    axes[0].grid(alpha=0.3)

    # Beam width
    beams = sorted(beam_results.keys())
    for m, c, mk in [("BLEU-4", "#e74c3c", "o"), ("METEOR", "#3498db", "s"), ("ROUGE-L", "#2ecc71", "^")]:
        vals = [beam_results[b][m] for b in beams]
        axes[1].plot(beams, vals, f"{mk}-", label=m, color=c, markersize=7, linewidth=2)
    axes[1].set_xlabel("Beam Width", fontsize=12)
    axes[1].set_ylabel("Score", fontsize=12)
    axes[1].set_title("Beam Width Sensitivity", fontsize=13, fontweight="bold")
    axes[1].legend(fontsize=10)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/task3_parameter_sensitivity.png", dpi=150)
    plt.show()
    print("[SAVED] task3_parameter_sensitivity.png")
else:
    print("[SKIP] No parameter sweep data found — run Task 2 first")


# =============================================================================
# Cell 8: FINE-TUNING COMPARISON (from Task 2)
# =============================================================================

if ft_results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    strategies = list(ft_results.keys())
    ft_colors = ["#2ecc71", "#3498db", "#e74c3c"]
    metric_names = ["BLEU-4", "METEOR", "ROUGE-L"]
    x = np.arange(len(metric_names))
    width = 0.25

    for i, (strat, color) in enumerate(zip(strategies, ft_colors)):
        m = ft_results[strat]["metrics"]
        vals = [m[mn] for mn in metric_names]
        bars = ax1.bar(x + i * width, vals, width, label=strat, color=color, edgecolor="white")
        for j, v in enumerate(vals):
            ax1.text(x[j] + i * width, v + 0.005, f"{v:.3f}", ha="center", fontsize=7)

    ax1.set_xticks(x + width)
    ax1.set_xticklabels(metric_names)
    ax1.set_ylabel("Score")
    ax1.set_title("Fine-Tuning Strategies: Metrics", fontweight="bold")
    ax1.legend()
    ax1.grid(axis="y", alpha=0.3)

    for strat, color in zip(strategies, ft_colors):
        losses = ft_results[strat]["losses"]
        ax2.plot(range(1, len(losses)+1), losses, "o-", label=strat, color=color, linewidth=2)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Training Loss")
    ax2.set_title("Training Loss Curves", fontweight="bold")
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/task3_finetuning_comparison.png", dpi=150)
    plt.show()
    print("[SAVED] task3_finetuning_comparison.png")
else:
    print("[SKIP] No fine-tuning data found — run Task 2 first")


# =============================================================================
# Cell 9: AUTO-GENERATED KEY OBSERVATIONS (for your report)
# =============================================================================

print("\n" + "=" * 70)
print("  KEY OBSERVATIONS (copy into your report)")
print("=" * 70)

# Decoding strategy observations
best_bleu1 = max(all_metrics, key=lambda s: all_metrics[s].get("BLEU-1", 0))
best_bleu4 = max(all_metrics, key=lambda s: all_metrics[s].get("BLEU-4", 0))
best_meteor = max(all_metrics, key=lambda s: all_metrics[s].get("METEOR", 0))
best_rouge = max(all_metrics, key=lambda s: all_metrics[s].get("ROUGE-L", 0))

print(f"""
1. DECODING STRATEGIES:
   - {best_bleu1} achieves highest BLEU-1 ({all_metrics[best_bleu1]['BLEU-1']:.4f}),
     indicating best unigram precision.
   - {best_bleu4} achieves highest BLEU-4 ({all_metrics[best_bleu4]['BLEU-4']:.4f}),
     indicating best 4-gram fluency.
   - {best_meteor} achieves highest METEOR ({all_metrics[best_meteor]['METEOR']:.4f}),
     which accounts for synonyms and stemming.
   - {best_rouge} achieves highest ROUGE-L ({all_metrics[best_rouge]['ROUGE-L']:.4f}),
     measuring longest common subsequence overlap.
""")

if temp_results:
    best_temp = max(temp_results, key=lambda t: temp_results[t].get("METEOR", 0))
    print(f"""2. TEMPERATURE SENSITIVITY:
   - Best temperature: {best_temp} (METEOR={temp_results[best_temp]['METEOR']:.4f})
   - Low temperatures (0.3) produce safe but repetitive captions.
   - High temperatures (1.5) produce diverse but sometimes incoherent captions.
   - Sweet spot is around {best_temp} for this model.
""")

if beam_results:
    best_beam = max(beam_results, key=lambda b: beam_results[b].get("BLEU-4", 0))
    print(f"""3. BEAM WIDTH:
   - Best beam width: {best_beam} (BLEU-4={beam_results[best_beam]['BLEU-4']:.4f})
   - Increasing beam width beyond {best_beam} shows diminishing returns.
   - Beam width 1 (greedy) underperforms wider beams across all metrics.
""")

if ft_results:
    best_ft = max(ft_results, key=lambda s: ft_results[s]["metrics"].get("METEOR", 0))
    print(f"""4. FINE-TUNING STRATEGIES:
   - {best_ft} achieves the best overall performance.
   - Prefix Tuning is most parameter-efficient but limited in capacity.
   - Full Fine-Tuning has the most parameters but risks overfitting.
   - LoRA offers a good balance between efficiency and performance.
""")


# =============================================================================
# Cell 10: SAVE FINAL TASK 3 RESULTS
# =============================================================================

task3_output = {
    "experiment": "milestone3_task3_visualization_comparison",
    "timestamp": datetime.now().isoformat(),
    "decoding_metrics": all_metrics,
    "caption_comparisons": comparison_data,
    "observations": {
        "best_bleu1": {"strategy": best_bleu1, "score": all_metrics[best_bleu1]["BLEU-1"]},
        "best_bleu4": {"strategy": best_bleu4, "score": all_metrics[best_bleu4]["BLEU-4"]},
        "best_meteor": {"strategy": best_meteor, "score": all_metrics[best_meteor]["METEOR"]},
        "best_rouge_l": {"strategy": best_rouge, "score": all_metrics[best_rouge]["ROUGE-L"]},
    },
}

save_path = f"{BASE_DIR}/task3_visualization_results.json"
with open(save_path, "w") as f:
    json.dump(task3_output, f, indent=2, default=str)

print(f"\n[SAVED] {save_path}")
print("\n" + "=" * 60)
print("  Task 3 Complete!")
print("=" * 60)
print("  Files generated:")
print("    - task3_decoding_comparison.png   (bar chart)")
print("    - task3_radar_chart.png           (radar/spider chart)")
print("    - task3_caption_comparison.png    (side-by-side captions)")
print("    - task3_parameter_sensitivity.png (temp + beam sweeps)")
print("    - task3_finetuning_comparison.png (FT bars + loss curves)")
print("    - task3_visualization_results.json (all results)")

print("\n" + "=" * 60)
print("  MILESTONE 3 FULLY COMPLETE!")
print("=" * 60)
print("""
  All files saved to Google Drive:
    /image-captioning/
      ├── task1_metrics_results.json
      ├── task2_parameter_results.json
      ├── task3_visualization_results.json
      ├── metrics_comparison.png
      ├── qualitative_evaluation.png
      ├── meteor_distribution.png
      ├── parameter_sweeps.png
      ├── finetuning_comparison.png
      ├── task3_decoding_comparison.png
      ├── task3_radar_chart.png
      ├── task3_caption_comparison.png
      ├── task3_parameter_sensitivity.png
      └── task3_finetuning_comparison.png
""")

In [ ]:
"""
RETRAIN WITH IMPROVED SETTINGS
- More epochs (20)
- Unfreeze last 4 GPT-2 layers
- Lower learning rate for stability
- 10 prefix tokens
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from pathlib import Path
from tqdm import tqdm
import json, random, time
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = Path("/content/drive/MyDrive/image-captioning/processed")

# ── Load data ──
tokenizer = GPT2Tokenizer.from_pretrained(str(SAVE_DIR / "tokenizer"), local_files_only=True)
data = torch.load(SAVE_DIR / "milestone1_tensors.pt", weights_only=True)

image_embeddings = data["image_embeddings"]
caption_input_ids = data["caption_input_ids"]
caption_attention_masks = data["caption_attention_masks"]

print(f"Data: {image_embeddings.shape[0]} pairs | Device: {DEVICE}")

# ── Dataset ──
class CaptionDataset(Dataset):
    def __init__(self, emb, ids, masks):
        self.emb, self.ids, self.masks = emb, ids, masks
    def __len__(self):
        return len(self.emb)
    def __getitem__(self, i):
        return self.emb[i], self.ids[i], self.masks[i]

random.seed(42)
idx = list(range(len(image_embeddings)))
random.shuffle(idx)
split = int(0.9 * len(idx))

train_ds = CaptionDataset(image_embeddings[idx[:split]], caption_input_ids[idx[:split]], caption_attention_masks[idx[:split]])
val_ds = CaptionDataset(image_embeddings[idx[split:]], caption_input_ids[idx[split:]], caption_attention_masks[idx[split:]])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

# ── Improved Model ──
class PrefixProjection(nn.Module):
    def __init__(self, clip_dim=512, gpt_dim=768, num_prefix=10):
        super().__init__()
        self.num_prefix = num_prefix
        self.gpt_dim = gpt_dim
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, gpt_dim * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(gpt_dim * 4, gpt_dim * num_prefix),
            nn.LayerNorm(gpt_dim * num_prefix),
        )
    def forward(self, x):
        return self.proj(x).view(-1, self.num_prefix, self.gpt_dim)

class ImprovedCaptionModel(nn.Module):
    def __init__(self, num_prefix=10, unfreeze_last_n=4):
        super().__init__()
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt2.resize_token_embeddings(len(tokenizer))
        gpt_dim = self.gpt2.config.n_embd
        self.prefix_proj = PrefixProjection(512, gpt_dim, num_prefix)
        self.num_prefix = num_prefix

        # Freeze all first
        for p in self.gpt2.parameters():
            p.requires_grad = False

        # Unfreeze last N transformer blocks
        for i in range(self.gpt2.config.n_layer - unfreeze_last_n, self.gpt2.config.n_layer):
            for p in self.gpt2.transformer.h[i].parameters():
                p.requires_grad = True

        # Unfreeze LM head, final layernorm, embeddings
        for p in self.gpt2.lm_head.parameters():
            p.requires_grad = True
        for p in self.gpt2.transformer.ln_f.parameters():
            p.requires_grad = True
        for p in self.gpt2.transformer.wte.parameters():
            p.requires_grad = True

        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Model: {total/1e6:.1f}M total | {trainable/1e6:.1f}M trainable ({100*trainable/total:.1f}%)")
        print(f"Prefix tokens: {num_prefix} | Unfrozen layers: {unfreeze_last_n}")

    def forward(self, clip_emb, input_ids, attention_mask):
        B, T = input_ids.shape
        prefix = self.prefix_proj(clip_emb)
        text_emb = self.gpt2.transformer.wte(input_ids)
        combined = torch.cat([prefix, text_emb], dim=1)

        prefix_mask = torch.ones(B, self.num_prefix, device=input_ids.device)
        full_mask = torch.cat([prefix_mask, attention_mask.float()], dim=1)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        prefix_labels = torch.full((B, self.num_prefix), -100, dtype=torch.long, device=input_ids.device)
        full_labels = torch.cat([prefix_labels, labels], dim=1)

        outputs = self.gpt2(inputs_embeds=combined, attention_mask=full_mask, labels=full_labels)
        return outputs.loss, outputs.logits

    @torch.no_grad()
    def generate(self, clip_emb, max_new_tokens=50, temperature=0.7, top_p=0.9):
        self.eval()
        if clip_emb.dim() == 1:
            clip_emb = clip_emb.unsqueeze(0)
        B = clip_emb.shape[0]
        device = clip_emb.device
        prefix = self.prefix_proj(clip_emb)
        cur = torch.full((B, 1), tokenizer.bos_token_id, device=device)

        for _ in range(max_new_tokens):
            text_emb = self.gpt2.transformer.wte(cur)
            embeds = torch.cat([prefix, text_emb], dim=1)
            logits = self.gpt2(inputs_embeds=embeds).logits[:, -1, :] / temperature

            sl, si = torch.sort(logits, descending=True)
            cum = torch.cumsum(F.softmax(sl, -1), -1)
            sl[cum - F.softmax(sl, -1) >= top_p] = -float("inf")
            nt = si.gather(-1, torch.multinomial(F.softmax(sl, -1), 1))

            cur = torch.cat([cur, nt], dim=1)
            if (nt == tokenizer.eos_token_id).all():
                break
        return [c.strip() for c in tokenizer.batch_decode(cur, skip_special_tokens=True)]

# Build model
model = ImprovedCaptionModel(num_prefix=10, unfreeze_last_n=4).to(DEVICE)

# ── Training ──
NUM_EPOCHS = 20
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-5, weight_decay=0.01
)

total_steps = NUM_EPOCHS * len(train_loader)
WARMUP = 200

def lr_lambda(step):
    if step < WARMUP:
        return step / WARMUP
    return max(0.05, 1.0 - (step - WARMUP) / (total_steps - WARMUP))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

best_val_loss = float("inf")
train_losses, val_losses = [], []
step = 0

# Sample images for tracking
sample_idx = random.sample(range(len(val_ds)), 5)
sample_embs = torch.stack([val_ds[i][0] for i in sample_idx]).to(DEVICE)

with open(SAVE_DIR / "milestone1_metadata.json") as f:
    meta = json.load(f)
sample_gts = [meta["pairs"][idx[split + i]]["caption"] for i in sample_idx]

print(f"\nTraining for {NUM_EPOCHS} epochs ({total_steps} steps)...")
print(f"{'=' * 60}\n")

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    # Train
    model.train()
    epoch_loss, n = 0, 0
    for emb, ids, mask in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        emb, ids, mask = emb.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()

        if scaler:
            with torch.amp.autocast("cuda"):
                loss, _ = model(emb, ids, mask)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss, _ = model(emb, ids, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()
        epoch_loss += loss.item()
        n += 1
        step += 1

    avg_train = epoch_loss / n

    # Validate
    model.eval()
    v_loss, vn = 0, 0
    with torch.no_grad():
        for emb, ids, mask in val_loader:
            emb, ids, mask = emb.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE)
            if scaler:
                with torch.amp.autocast("cuda"):
                    loss, _ = model(emb, ids, mask)
            else:
                loss, _ = model(emb, ids, mask)
            v_loss += loss.item()
            vn += 1
    avg_val = v_loss / vn

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    elapsed = time.time() - t0

    print(f"  Train: {avg_train:.4f} | Val: {avg_val:.4f} | Time: {elapsed:.0f}s")

    # Save best
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        save_dir = Path("/content/drive/MyDrive/image-captioning/models")
        save_dir.mkdir(parents=True, exist_ok=True)
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_loss": avg_val,
            "config": {"num_prefix": 10, "unfreeze_last_n": 4},
        }, save_dir / "best_model_v2.pt")
        print(f"  ** New best saved (val_loss: {avg_val:.4f})")

    # Show sample captions every 5 epochs
    if (epoch + 1) % 5 == 0:
        caps = model.generate(sample_embs, max_new_tokens=40)
        print(f"\n  Sample captions (Epoch {epoch+1}):")
        for i in range(min(3, len(caps))):
            print(f"    GT:  {sample_gts[i][:60]}")
            print(f"    Gen: {caps[i][:60]}\n")

print(f"\n{'=' * 60}")
print(f"Done! Best val loss: {best_val_loss:.4f}")
print(f"{'=' * 60}")

# ── Plot ──
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, NUM_EPOCHS+1), train_losses, "b-o", label="Train", markersize=4)
ax.plot(range(1, NUM_EPOCHS+1), val_losses, "r-o", label="Val", markersize=4)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Improved Model — Training & Validation Loss")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/image-captioning/loss_curves_v2.png", dpi=150)
plt.show()
print("[SAVED] loss_curves_v2.png")

In [ ]:
from PIL import Image
from google.colab import files, drive
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer, CLIPModel, CLIPProcessor
from pathlib import Path

drive.mount('/content/drive', force_remount=False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load tokenizer ──
SAVE_DIR = Path("/content/drive/MyDrive/image-captioning/processed")
tokenizer = GPT2Tokenizer.from_pretrained(str(SAVE_DIR / "tokenizer"), local_files_only=True)

# ── Load CLIP ──
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

@torch.no_grad()
def encode_image(image):
    inputs = clip_processor(images=image, return_tensors="pt").to(DEVICE)
    outputs = clip_model.get_image_features(**inputs)
    if isinstance(outputs, torch.Tensor):
        feats = outputs
    else:
        feats = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs[0]
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats

# ── Model definition ──
class PrefixProjection(nn.Module):
    def __init__(self, clip_dim=512, gpt_dim=768, num_prefix_tokens=8):
        super().__init__()
        self.num_prefix_tokens = num_prefix_tokens
        self.gpt_dim = gpt_dim
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, gpt_dim * 2), nn.GELU(),
            nn.Linear(gpt_dim * 2, gpt_dim * num_prefix_tokens),
            nn.LayerNorm(gpt_dim * num_prefix_tokens),
        )
    def forward(self, x):
        return self.proj(x).view(-1, self.num_prefix_tokens, self.gpt_dim)

class CLIPGpt2CaptionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt2.resize_token_embeddings(len(tokenizer))
        self.prefix_proj = PrefixProjection(512, self.gpt2.config.n_embd, 8)
        self.num_prefix = 8
        for p in self.gpt2.parameters():
            p.requires_grad = False

    def _generate(self, clip_emb, max_new_tokens, next_token_fn):
        self.eval()
        if clip_emb.dim() == 1: clip_emb = clip_emb.unsqueeze(0)
        prefix = self.prefix_proj(clip_emb)
        cur = torch.full((1, 1), tokenizer.bos_token_id, device=clip_emb.device)
        for _ in range(max_new_tokens):
            embeds = torch.cat([prefix, self.gpt2.transformer.wte(cur)], dim=1)
            logits = self.gpt2(inputs_embeds=embeds).logits[:, -1, :]
            nt = next_token_fn(logits)
            cur = torch.cat([cur, nt], dim=1)
            if (nt == tokenizer.eos_token_id).all(): break
        return [c.strip() for c in tokenizer.batch_decode(cur, skip_special_tokens=True)]

    @torch.no_grad()
    def generate_greedy(self, clip_emb, max_new_tokens=40):
        return self._generate(clip_emb, max_new_tokens, lambda l: l.argmax(-1, keepdim=True))

    @torch.no_grad()
    def generate_nucleus(self, clip_emb, max_new_tokens=40, temperature=0.7, top_p=0.9):
        def sample(logits):
            logits = logits / temperature
            sl, si = torch.sort(logits, descending=True)
            cum = torch.cumsum(F.softmax(sl, -1), -1)
            sl[cum - F.softmax(sl, -1) >= top_p] = -float("inf")
            return si.gather(-1, torch.multinomial(F.softmax(sl, -1), 1))
        return self._generate(clip_emb, max_new_tokens, sample)

    @torch.no_grad()
    def generate_beam(self, clip_emb, max_new_tokens=40, num_beams=5):
        self.eval()
        if clip_emb.dim() == 1: clip_emb = clip_emb.unsqueeze(0)
        device = clip_emb.device
        prefix = self.prefix_proj(clip_emb)
        beams = [(torch.tensor([[tokenizer.bos_token_id]], device=device), 0.0)]
        completed = []
        for _ in range(max_new_tokens):
            cands = []
            for seq, score in beams:
                embeds = torch.cat([prefix, self.gpt2.transformer.wte(seq)], dim=1)
                lp = F.log_softmax(self.gpt2(inputs_embeds=embeds).logits[:, -1, :], -1)
                top_lp, top_ids = lp.topk(num_beams, -1)
                for i in range(num_beams):
                    tid = top_ids[0,i].item()
                    ns = score + top_lp[0,i].item()
                    nseq = torch.cat([seq, torch.tensor([[tid]], device=device)], 1)
                    (completed if tid == tokenizer.eos_token_id else cands).append((nseq, ns))
            if not cands: break
            cands.sort(key=lambda x: x[1], reverse=True)
            beams = cands[:num_beams]
        opts = completed + beams
        opts.sort(key=lambda x: x[1]/x[0].size(1), reverse=True)
        return tokenizer.batch_decode(opts[0][0], skip_special_tokens=True)

# ── Load trained weights ──
model = CLIPGpt2CaptionModel().to(DEVICE)
ckpt = torch.load("/content/drive/MyDrive/image-captioning/models/best_model.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"], strict=False)
model.eval()
print(f"[INFO] Model loaded! (epoch {ckpt['epoch']+1}, val_loss {ckpt['val_loss']:.4f})")

# ── Upload and caption your image ──
print("\nChoose an image to caption:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
img = Image.open(filename).convert("RGB")

clip_emb = encode_image(img)
cap_greedy = model.generate_greedy(clip_emb)[0]
cap_beam = model.generate_beam(clip_emb, num_beams=5)[0]
cap_nucleus = model.generate_nucleus(clip_emb, temperature=0.7, top_p=0.9)[0]

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis("off")
plt.title(filename, fontsize=12)
plt.show()

print(f"\nGreedy:  {cap_greedy}")
print(f"Beam:    {cap_beam}")
print(f"Nucleus: {cap_nucleus}")

In [ ]:
from google.colab import files
import time

# Upload multiple images
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n{'=' * 50}")
    print(f"  {filename}")
    print(f"{'=' * 50}")

    start = time.time()
    img = Image.open(filename).convert("RGB")
    clip_emb = encode_image(img)

    cap_greedy = model.generate_greedy(clip_emb)[0]
    cap_beam = model.generate_beam(clip_emb, num_beams=5)[0]
    cap_nucleus = model.generate_nucleus(clip_emb, temperature=0.7, top_p=0.9)[0]
    elapsed = time.time() - start

    plt.figure(figsize=(6, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(filename, fontsize=11)
    plt.show()

    print(f"  Greedy:  {cap_greedy}")
    print(f"  Beam:    {cap_beam}")
    print(f"  Nucleus: {cap_nucleus}")
    print(f"  Time:    {elapsed:.2f}s")

In [ ]:
from PIL import Image
from google.colab import files
import matplotlib.pyplot as plt
import time

# Load the new improved model
new_ckpt = torch.load("/content/drive/MyDrive/image-captioning/models/best_model_v2.pt", map_location=DEVICE)
model.load_state_dict(new_ckpt["model_state_dict"])
model.eval()
print(f"Loaded improved model (epoch {new_ckpt['epoch']+1}, val_loss {new_ckpt['val_loss']:.4f})")

# Upload and caption
print("\nUpload an image:")
uploaded = files.upload()

for filename in uploaded.keys():
    img = Image.open(filename).convert("RGB")
    clip_emb = encode_image(img)

    start = time.time()
    cap1 = model.generate(clip_emb, temperature=0.7, top_p=0.9)[0]
    cap2 = model.generate(clip_emb, temperature=0.5, top_p=0.85)[0]
    cap3 = model.generate(clip_emb, temperature=0.3, top_p=0.8)[0]
    elapsed = time.time() - start

    plt.figure(figsize=(6, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(filename, fontsize=11)
    plt.show()

    print(f"\n  Creative (t=0.7):   {cap1}")
    print(f"  Balanced (t=0.5):   {cap2}")
    print(f"  Confident (t=0.3):  {cap3}")
    print(f"  Time: {elapsed:.2f}s")

In [ ]:
"""
=============================================================================
TRAIN ON 20K COCO IMAGES — FIXED
Skips file existence checks (Drive is flaky), handles errors during encoding
~45 min total on T4 GPU
=============================================================================
"""

# ═══════════════════════════════════════════════════════════════════
# CELL 1: Setup
# ═══════════════════════════════════════════════════════════════════

import os, json, random, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor, GPT2LMHeadModel, GPT2Tokenizer
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from pycocotools.coco import COCO
import matplotlib.pyplot as plt
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount('/content/drive', force_remount=False)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/image-captioning/data")
IMG_DIR = DATA_DIR / "train2017"
ANN_FILE = DATA_DIR / "annotations" / "captions_train2017.json"
SAVE_DIR = Path("/content/drive/MyDrive/image-captioning/processed")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")

# Quick Drive test
try:
    test_files = os.listdir(IMG_DIR)
    print(f"Drive OK! Images in folder: {len(test_files)}")
except Exception as e:
    print(f"WARNING: Drive issue: {e}")
    print("Try: Runtime > Restart runtime, then remount")


# ═══════════════════════════════════════════════════════════════════
# CELL 2: Build 20k pairs (NO file existence check)
# ═══════════════════════════════════════════════════════════════════

coco = COCO(str(ANN_FILE))
all_img_ids = list(coco.imgs.keys())
print(f"Total COCO images: {len(all_img_ids)}")

random.seed(SEED)
sampled_ids = sorted(random.sample(all_img_ids, 20000))

print("Building 20k pairs (skipping file checks)...")
pairs = []

for img_id in sampled_ids:
    info = coco.imgs[img_id]
    fname = info["file_name"]
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)
    if not anns:
        continue
    caption = anns[0]["caption"].strip().lower()
    pairs.append({
        "image_id": img_id,
        "path": str(IMG_DIR / fname),
        "caption": caption,
    })

print(f"Pairs built: {len(pairs)}")
print(f"Example: {pairs[0]['caption']}")

# Verify one image loads
try:
    test_img = Image.open(pairs[0]["path"]).convert("RGB")
    print(f"Test image OK: {test_img.size}")
except Exception as e:
    print(f"ERROR reading image: {e}")
    print("Restart runtime and remount Drive before continuing")


# ═══════════════════════════════════════════════════════════════════
# CELL 3: Encode with CLIP (~10 min) — skips bad images
# ═══════════════════════════════════════════════════════════════════

EMB_PATH = SAVE_DIR / "embeddings_20k.pt"

if EMB_PATH.exists():
    print("Embeddings already exist, loading...")
    saved = torch.load(EMB_PATH, weights_only=True)
    all_embeddings = saved["embeddings"]
    valid_indices = saved["valid_indices"]
    pairs = [pairs[i] for i in valid_indices]
    print(f"Loaded: {all_embeddings.shape} ({len(pairs)} valid pairs)")
else:
    print(f"Encoding {len(pairs)} images with CLIP...")
    print("Bad images will be skipped automatically.\n")

    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model.eval()

    all_embeddings = []
    valid_indices = []
    failed = 0

    for i in tqdm(range(len(pairs)), desc="CLIP encoding"):
        try:
            img = Image.open(pairs[i]["path"]).convert("RGB")
            with torch.no_grad():
                inputs = clip_processor(images=img, return_tensors="pt").to(DEVICE)
                outputs = clip_model.get_image_features(**inputs)
                if isinstance(outputs, torch.Tensor):
                    feats = outputs
                else:
                    feats = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs[0]
                feats = feats / feats.norm(dim=-1, keepdim=True)
                all_embeddings.append(feats.cpu().squeeze(0))
                valid_indices.append(i)
        except Exception:
            failed += 1
            continue

        # Save checkpoint every 5000 images
        if len(all_embeddings) % 5000 == 0 and len(all_embeddings) > 0:
            print(f"  Checkpoint: {len(all_embeddings)} encoded, {failed} failed")

    all_embeddings = torch.stack(all_embeddings)
    pairs = [pairs[i] for i in valid_indices]

    torch.save({"embeddings": all_embeddings, "valid_indices": valid_indices}, EMB_PATH)
    print(f"\nDone! {all_embeddings.shape} embeddings saved ({failed} failed)")

    del clip_model, clip_processor
    torch.cuda.empty_cache()


# ═══════════════════════════════════════════════════════════════════
# CELL 4: Tokenize captions
# ═══════════════════════════════════════════════════════════════════

TOK_PATH = SAVE_DIR / "tokens_20k.pt"

tok_dir = SAVE_DIR / "tokenizer"
if tok_dir.exists():
    tokenizer = GPT2Tokenizer.from_pretrained(str(tok_dir), local_files_only=True)
    print(f"Tokenizer loaded: vocab {len(tokenizer)}")
else:
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.add_special_tokens({
        "bos_token": "<|startofcaption|>",
        "eos_token": "<|endofcaption|>",
        "pad_token": "<|pad|>",
    })
    tokenizer.save_pretrained(str(tok_dir))
    print(f"Tokenizer created: vocab {len(tokenizer)}")

if TOK_PATH.exists():
    print("Tokens already exist, loading...")
    tok_data = torch.load(TOK_PATH, weights_only=True)
    caption_ids = tok_data["input_ids"]
    caption_masks = tok_data["attention_masks"]
    print(f"Loaded: {caption_ids.shape}")
else:
    print(f"Tokenizing {len(pairs)} captions...")
    MAX_LEN = 64
    all_ids, all_masks = [], []

    for p in tqdm(pairs, desc="Tokenizing"):
        text = f"{tokenizer.bos_token} {p['caption']} {tokenizer.eos_token}"
        enc = tokenizer(text, max_length=MAX_LEN, padding="max_length",
                        truncation=True, return_tensors="pt")
        all_ids.append(enc["input_ids"].squeeze(0))
        all_masks.append(enc["attention_mask"].squeeze(0))

    caption_ids = torch.stack(all_ids)
    caption_masks = torch.stack(all_masks)
    torch.save({"input_ids": caption_ids, "attention_masks": caption_masks}, TOK_PATH)
    print(f"Saved: {caption_ids.shape}")

# Match sizes
min_len = min(len(all_embeddings), len(caption_ids))
all_embeddings = all_embeddings[:min_len]
caption_ids = caption_ids[:min_len]
caption_masks = caption_masks[:min_len]
pairs = pairs[:min_len]

print(f"\nDataset ready: {min_len} pairs")


# ═══════════════════════════════════════════════════════════════════
# CELL 5: DataLoader
# ═══════════════════════════════════════════════════════════════════

class CaptionDataset(Dataset):
    def __init__(self, emb, ids, masks):
        self.emb, self.ids, self.masks = emb, ids, masks
    def __len__(self):
        return len(self.emb)
    def __getitem__(self, i):
        return self.emb[i], self.ids[i], self.masks[i]

N = len(all_embeddings)
idx = list(range(N))
random.shuffle(idx)
split = int(0.9 * N)

train_ds = CaptionDataset(all_embeddings[idx[:split]], caption_ids[idx[:split]], caption_masks[idx[:split]])
val_ds = CaptionDataset(all_embeddings[idx[split:]], caption_ids[idx[split:]], caption_masks[idx[split:]])

train_loader = DataLoader(train_ds, batch_size=48, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=48, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")
print(f"Batches per epoch: {len(train_loader)}")


# ═══════════════════════════════════════════════════════════════════
# CELL 6: Model
# ═══════════════════════════════════════════════════════════════════

class PrefixProjection(nn.Module):
    def __init__(self, clip_dim=512, gpt_dim=768, num_prefix=10):
        super().__init__()
        self.num_prefix = num_prefix
        self.gpt_dim = gpt_dim
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, gpt_dim * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(gpt_dim * 4, gpt_dim * num_prefix),
            nn.LayerNorm(gpt_dim * num_prefix),
        )
    def forward(self, x):
        return self.proj(x).view(-1, self.num_prefix, self.gpt_dim)

class CaptionModel(nn.Module):
    def __init__(self, num_prefix=10, unfreeze_last_n=4):
        super().__init__()
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt2.resize_token_embeddings(len(tokenizer))
        gpt_dim = self.gpt2.config.n_embd
        self.prefix_proj = PrefixProjection(512, gpt_dim, num_prefix)
        self.num_prefix = num_prefix

        for p in self.gpt2.parameters():
            p.requires_grad = False
        for i in range(self.gpt2.config.n_layer - unfreeze_last_n, self.gpt2.config.n_layer):
            for p in self.gpt2.transformer.h[i].parameters():
                p.requires_grad = True
        for p in self.gpt2.lm_head.parameters():
            p.requires_grad = True
        for p in self.gpt2.transformer.ln_f.parameters():
            p.requires_grad = True
        for p in self.gpt2.transformer.wte.parameters():
            p.requires_grad = True

        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Model: {total/1e6:.1f}M total | {trainable/1e6:.1f}M trainable ({100*trainable/total:.1f}%)")

    def forward(self, clip_emb, input_ids, attention_mask):
        B, T = input_ids.shape
        prefix = self.prefix_proj(clip_emb)
        text_emb = self.gpt2.transformer.wte(input_ids)
        combined = torch.cat([prefix, text_emb], dim=1)

        prefix_mask = torch.ones(B, self.num_prefix, device=input_ids.device)
        full_mask = torch.cat([prefix_mask, attention_mask.float()], dim=1)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        prefix_labels = torch.full((B, self.num_prefix), -100, dtype=torch.long, device=input_ids.device)
        full_labels = torch.cat([prefix_labels, labels], dim=1)

        outputs = self.gpt2(inputs_embeds=combined, attention_mask=full_mask, labels=full_labels)
        return outputs.loss, outputs.logits

    @torch.no_grad()
    def generate(self, clip_emb, max_new_tokens=50, temperature=0.7, top_p=0.9):
        self.eval()
        if clip_emb.dim() == 1:
            clip_emb = clip_emb.unsqueeze(0)
        prefix = self.prefix_proj(clip_emb)
        cur = torch.full((clip_emb.shape[0], 1), tokenizer.bos_token_id, device=clip_emb.device)
        for _ in range(max_new_tokens):
            embeds = torch.cat([prefix, self.gpt2.transformer.wte(cur)], dim=1)
            logits = self.gpt2(inputs_embeds=embeds).logits[:, -1, :] / temperature
            sl, si = torch.sort(logits, descending=True)
            cum = torch.cumsum(F.softmax(sl, -1), -1)
            sl[cum - F.softmax(sl, -1) >= top_p] = -float("inf")
            nt = si.gather(-1, torch.multinomial(F.softmax(sl, -1), 1))
            cur = torch.cat([cur, nt], dim=1)
            if (nt == tokenizer.eos_token_id).all():
                break
        return [c.strip() for c in tokenizer.batch_decode(cur, skip_special_tokens=True)]

model = CaptionModel(num_prefix=10, unfreeze_last_n=4).to(DEVICE)


# ═══════════════════════════════════════════════════════════════════
# CELL 7: Train (15 epochs, ~30 min on T4)
# ═══════════════════════════════════════════════════════════════════

NUM_EPOCHS = 15
LR = 3e-5
WARMUP = 300

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=0.01
)

total_steps = NUM_EPOCHS * len(train_loader)

def lr_lambda(step):
    if step < WARMUP:
        return step / WARMUP
    return max(0.05, 1.0 - (step - WARMUP) / (total_steps - WARMUP))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

best_val_loss = float("inf")
train_losses, val_losses = [], []
step = 0

sample_idx = random.sample(range(len(val_ds)), 5)
sample_embs = torch.stack([val_ds[i][0] for i in sample_idx]).to(DEVICE)
sample_gts = [pairs[idx[split + i]]["caption"] for i in sample_idx]

print(f"\nTraining: {NUM_EPOCHS} epochs x {len(train_loader)} batches")
print(f"Total steps: {total_steps:,}")
print(f"Estimated time: ~30 min on T4")
print(f"{'=' * 60}\n")

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    model.train()
    epoch_loss, n = 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for emb, ids, mask in pbar:
        emb, ids, mask = emb.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()

        if scaler:
            with torch.amp.autocast("cuda"):
                loss, _ = model(emb, ids, mask)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss, _ = model(emb, ids, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()
        epoch_loss += loss.item()
        n += 1
        step += 1

        if step % 100 == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train = epoch_loss / n

    model.eval()
    v_loss, vn = 0, 0
    with torch.no_grad():
        for emb, ids, mask in val_loader:
            emb, ids, mask = emb.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE)
            if scaler:
                with torch.amp.autocast("cuda"):
                    loss, _ = model(emb, ids, mask)
            else:
                loss, _ = model(emb, ids, mask)
            v_loss += loss.item()
            vn += 1
    avg_val = v_loss / vn

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    elapsed = time.time() - t0

    print(f"  Train={avg_train:.4f} | Val={avg_val:.4f} | Time={elapsed:.0f}s")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        save_dir = Path("/content/drive/MyDrive/image-captioning/models")
        save_dir.mkdir(parents=True, exist_ok=True)
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "val_loss": avg_val,
            "config": {"num_prefix": 10, "unfreeze_last_n": 4, "data": "20k"},
        }, save_dir / "best_model_20k.pt")
        print(f"  ** Best model saved! (val_loss: {avg_val:.4f})")

    if (epoch + 1) % 3 == 0:
        caps = model.generate(sample_embs, max_new_tokens=40)
        print(f"\n  Sample captions:")
        for i in range(min(3, len(caps))):
            print(f"    GT:  {sample_gts[i][:60]}")
            print(f"    Gen: {caps[i][:60]}\n")

print(f"\n{'=' * 60}")
print(f"Done! Best val loss: {best_val_loss:.4f}")
print(f"Model saved: best_model_20k.pt")
print(f"{'=' * 60}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, NUM_EPOCHS+1), train_losses, "b-o", label="Train", markersize=4)
ax.plot(range(1, NUM_EPOCHS+1), val_losses, "r-o", label="Val", markersize=4)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("20k COCO — Training & Validation Loss")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/image-captioning/loss_curves_20k.png", dpi=150)
plt.show()

In [ ]:
from PIL import Image
from google.colab import files, drive
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer, CLIPModel, CLIPProcessor
from pathlib import Path
import time

drive.mount('/content/drive', force_remount=False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer
SAVE_DIR = Path("/content/drive/MyDrive/image-captioning/processed")
tokenizer = GPT2Tokenizer.from_pretrained(str(SAVE_DIR / "tokenizer"), local_files_only=True)

# Load CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

@torch.no_grad()
def encode_image(image):
    inputs = clip_processor(images=image, return_tensors="pt").to(DEVICE)
    outputs = clip_model.get_image_features(**inputs)
    feats = outputs if isinstance(outputs, torch.Tensor) else (outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs[0])
    return feats / feats.norm(dim=-1, keepdim=True)

# Model
class PrefixProjection(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_prefix, self.gpt_dim = 10, 768
        self.proj = nn.Sequential(
            nn.Linear(512, 768*4), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(768*4, 768*10), nn.LayerNorm(768*10))
    def forward(self, x):
        return self.proj(x).view(-1, 10, 768)

class CaptionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt2.resize_token_embeddings(len(tokenizer))
        self.prefix_proj = PrefixProjection()
        self.num_prefix = 10
        for p in self.gpt2.parameters(): p.requires_grad = False

    @torch.no_grad()
    def generate(self, clip_emb, max_new_tokens=50, temperature=0.7, top_p=0.9):
        self.eval()
        if clip_emb.dim() == 1: clip_emb = clip_emb.unsqueeze(0)
        prefix = self.prefix_proj(clip_emb)
        cur = torch.full((1, 1), tokenizer.bos_token_id, device=clip_emb.device)
        for _ in range(max_new_tokens):
            embeds = torch.cat([prefix, self.gpt2.transformer.wte(cur)], dim=1)
            logits = self.gpt2(inputs_embeds=embeds).logits[:, -1, :] / temperature
            sl, si = torch.sort(logits, descending=True)
            cum = torch.cumsum(F.softmax(sl, -1), -1)
            sl[cum - F.softmax(sl, -1) >= top_p] = -float("inf")
            nt = si.gather(-1, torch.multinomial(F.softmax(sl, -1), 1))
            cur = torch.cat([cur, nt], dim=1)
            if (nt == tokenizer.eos_token_id).all(): break
        return [c.strip() for c in tokenizer.batch_decode(cur, skip_special_tokens=True)]

# Load trained weights
model = CaptionModel().to(DEVICE)
ckpt = torch.load("/content/drive/MyDrive/image-captioning/models/best_model_20k.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"], strict=False)
model.eval()
print(f"Model loaded! (epoch {ckpt['epoch']+1}, val_loss {ckpt['val_loss']:.4f})")

# ── Upload & Caption ──
print("\nUpload images:")
uploaded = files.upload()

for filename in uploaded.keys():
    img = Image.open(filename).convert("RGB")
    clip_emb = encode_image(img)

    start = time.time()
    cap1 = model.generate(clip_emb, temperature=0.7, top_p=0.9)[0]
    cap2 = model.generate(clip_emb, temperature=0.5, top_p=0.85)[0]
    cap3 = model.generate(clip_emb, temperature=0.3, top_p=0.8)[0]
    elapsed = time.time() - start

    plt.figure(figsize=(6, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(filename, fontsize=11)
    plt.show()

    print(f"\n  Creative (t=0.7):   {cap1}")
    print(f"  Balanced (t=0.5):   {cap2}")
    print(f"  Confident (t=0.3):  {cap3}")
    print(f"  Time: {elapsed:.2f}s\n")

In [ ]:
"""
=============================================================================
DEMO DAY — IMAGE CAPTIONING PIPELINE
One cell. Loads everything. Upload images. Get captions. No errors.
=============================================================================
"""

# ── Step 1: All imports ──
from PIL import Image
from google.colab import files, drive
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer, CLIPModel, CLIPProcessor
from pathlib import Path
import time

# ── Step 2: Mount Drive ──
drive.mount('/content/drive', force_remount=False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Step 3: Load Tokenizer ──
SAVE_DIR = Path("/content/drive/MyDrive/image-captioning/processed")
tokenizer = GPT2Tokenizer.from_pretrained(str(SAVE_DIR / "tokenizer"), local_files_only=True)

# ── Step 4: Load CLIP ──
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

@torch.no_grad()
def encode_image(image):
    inputs = clip_processor(images=image, return_tensors="pt").to(DEVICE)
    outputs = clip_model.get_image_features(**inputs)
    feats = outputs if isinstance(outputs, torch.Tensor) else (outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs[0])
    return feats / feats.norm(dim=-1, keepdim=True)

# ── Step 5: Define Model ──
class PrefixProjection(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_prefix, self.gpt_dim = 10, 768
        self.proj = nn.Sequential(
            nn.Linear(512, 768 * 4), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(768 * 4, 768 * 10), nn.LayerNorm(768 * 10))
    def forward(self, x):
        return self.proj(x).view(-1, 10, 768)

class CaptionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt2.resize_token_embeddings(len(tokenizer))
        self.prefix_proj = PrefixProjection()
        self.num_prefix = 10
        for p in self.gpt2.parameters():
            p.requires_grad = False

    def _gen(self, clip_emb, max_tokens, get_next):
        self.eval()
        if clip_emb.dim() == 1: clip_emb = clip_emb.unsqueeze(0)
        prefix = self.prefix_proj(clip_emb)
        cur = torch.full((1, 1), tokenizer.bos_token_id, device=clip_emb.device)
        for _ in range(max_tokens):
            embeds = torch.cat([prefix, self.gpt2.transformer.wte(cur)], dim=1)
            logits = self.gpt2(inputs_embeds=embeds).logits[:, -1, :]
            nt = get_next(logits)
            cur = torch.cat([cur, nt], dim=1)
            if (nt == tokenizer.eos_token_id).all(): break
        return [c.strip() for c in tokenizer.batch_decode(cur, skip_special_tokens=True)]

    @torch.no_grad()
    def generate_greedy(self, clip_emb, max_new_tokens=40):
        return self._gen(clip_emb, max_new_tokens,
            lambda l: l.argmax(-1, keepdim=True))

    @torch.no_grad()
    def generate_nucleus(self, clip_emb, max_new_tokens=40, temperature=0.8, top_p=0.9):
        def sample(logits):
            logits = logits / temperature
            sl, si = torch.sort(logits, descending=True)
            cum = torch.cumsum(F.softmax(sl, -1), -1)
            sl[cum - F.softmax(sl, -1) >= top_p] = -float("inf")
            return si.gather(-1, torch.multinomial(F.softmax(sl, -1), 1))
        return self._gen(clip_emb, max_new_tokens, sample)

    @torch.no_grad()
    def generate_beam(self, clip_emb, max_new_tokens=40, num_beams=5):
        self.eval()
        if clip_emb.dim() == 1: clip_emb = clip_emb.unsqueeze(0)
        device = clip_emb.device
        prefix = self.prefix_proj(clip_emb)
        beams = [(torch.tensor([[tokenizer.bos_token_id]], device=device), 0.0)]
        completed = []
        for _ in range(max_new_tokens):
            cands = []
            for seq, score in beams:
                embeds = torch.cat([prefix, self.gpt2.transformer.wte(seq)], dim=1)
                lp = F.log_softmax(self.gpt2(inputs_embeds=embeds).logits[:, -1, :], -1)
                top_lp, top_ids = lp.topk(num_beams, -1)
                for i in range(num_beams):
                    tid = top_ids[0, i].item()
                    ns = score + top_lp[0, i].item()
                    nseq = torch.cat([seq, torch.tensor([[tid]], device=device)], 1)
                    (completed if tid == tokenizer.eos_token_id else cands).append((nseq, ns))
            if not cands: break
            cands.sort(key=lambda x: x[1], reverse=True)
            beams = cands[:num_beams]
        opts = completed + beams
        opts.sort(key=lambda x: x[1] / x[0].size(1), reverse=True)
        return tokenizer.batch_decode(opts[0][0], skip_special_tokens=True)

# ── Step 6: Load Trained Weights ──
model = CaptionModel().to(DEVICE)

# Try 20k model first, fall back to original
MODEL_PATH = "/content/drive/MyDrive/image-captioning/models/best_model_20k.pt"
if not Path(MODEL_PATH).exists():
    MODEL_PATH = "/content/drive/MyDrive/image-captioning/models/best_model.pt"

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"], strict=False)
model.eval()

print("=" * 55)
print("  IMAGE CAPTIONING — READY FOR DEMO")
print(f"  Model: epoch {ckpt['epoch']+1} | val_loss: {ckpt['val_loss']:.4f}")
print(f"  Device: {DEVICE}")
print(f"  Checkpoint: {Path(MODEL_PATH).name}")
print("=" * 55)

# ── Step 7: Upload & Caption (runs in a loop) ──
while True:
    print("\nUpload image(s) — click Cancel to stop:")
    try:
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded. Done!")
            break

        for filename in uploaded.keys():
            img = Image.open(filename).convert("RGB")
            clip_emb = encode_image(img)

            start = time.time()
            cap_greedy = model.generate_greedy(clip_emb)[0]
            cap_beam = model.generate_beam(clip_emb, num_beams=5)[0]
            cap_nucleus = model.generate_nucleus(clip_emb, temperature=0.8, top_p=0.9)[0]
            elapsed = time.time() - start

            plt.figure(figsize=(8, 6))
            plt.imshow(img)
            plt.axis("off")
            plt.title(filename, fontsize=12, fontweight="bold")
            plt.show()

            print(f"\n  Greedy:        {cap_greedy}")
            print(f"  Beam Search:   {cap_beam}")
            print(f"  Nucleus:       {cap_nucleus}")
            print(f"  Time:          {elapsed:.2f}s")
            print(f"  {'─' * 45}")

    except Exception as e:
        if "cancel" in str(e).lower() or "keyboard" in str(e).lower():
            print("\nDemo ended!")
            break
        else:
            print(f"\nError: {e}")
            print("Skipping... try another image.\n")
            continue

print("\nDone! Thanks for watching the demo.")